# ArcFace — huấn luyện lại S3 (Nôm) trên Kaggle · **self-contained** · reset-safe

Notebook **tự chứa mã nguồn** (cell *Ghi mã nguồn* ghi các module ra `/kaggle/working/arcface`),
nên bạn **chỉ cần gắn DATA**, không cần upload code.

**Trước khi chạy:** Settings → **GPU T4 x2** + **Internet ON** · Add-ons → Secrets → thêm `HF_TOKEN` (quyền write) ·
Add Data → gắn dataset chứa `manifest.csv` + `crops/` (đã chạy `prepare_data.py` ở máy).

Reset/hết giờ → **Run All** lại: cell Train tự **resume từ HF**, chỉ chạy epoch còn thiếu.

### 1) Cấu hình — sửa ở đây

In [ ]:
HF_REPO = "mdnt571/nom-embed-arcface"   # ← BẮT BUỘC để reset-safe (đẩy+resume). "" = local-only.
EPOCHS  = 30
BATCH   = 128
K       = 3               # sub-centers ArcFace (chịu nhãn nhiễu)
SPLIT   = "page_disjoint" # hoặc "lobo" (bỏ 1 sách ra test)
HOLDOUT = ""              # ví dụ "stt4" khi SPLIT="lobo"
SAMPLER = "balanced"      # hoặc "confusion" (hard-negative theo chữ giống)
USE_SAM = True
USE_SWA = False

### 2) Ghi mã nguồn ArcFace ra `/kaggle/working/arcface` (tự chứa)

In [ ]:
import base64, os
CODE_DIR = "/kaggle/working/arcface" if os.path.isdir("/kaggle/working") else "arcface"
os.makedirs(CODE_DIR, exist_ok=True)
MODULES = {
    "model.py": "IiIiTW9kZWw6IE5vbUVtYmVkZGVyIGJhY2tib25lIChjaGVja3BvaW50LWNvbXBhdGlibGUpICsgU3ViLWNlbnRlciBBcmNGYWNlIGhlYWQuCgpUaGUgQkFDS0JPTkUgaXMgYnl0ZS1pZGVudGljYWwgaW4gc3RydWN0dXJlIHRvCnBpcGVsaW5lL2FsaWduX2VuZ2luZS9ub21fY2xhc3NpZmllci9tb2RlbC5weSBzbyBhIGNoZWNrcG9pbnQgZXhwb3J0ZWQgaGVyZSBpcyBhCmRyb3AtaW4gZm9yIHRoZSByZXBvJ3MgTm9tRW5jb2RlciAoaW5mZXIucHkgbG9hZHMgY2tbImJhY2tib25lIl0gaW50byB0aGlzIGV4YWN0Cm1vZHVsZSkuIFRoZSBIRUFEIGlzIHVwZ3JhZGVkIHRvICoqU3ViLWNlbnRlciBBcmNGYWNlKiogKERlbmcgZXQgYWwuLCBFQ0NWIDIwMjAsCiJTdWItY2VudGVyIEFyY0ZhY2U6IEJvb3N0aW5nIEZhY2UgUmVjb2duaXRpb24gYnkgTGFyZ2UtU2NhbGUgTm9pc3kgV2ViIEZhY2VzIik6Cksgc3ViLWNlbnRlcnMgcGVyIGNsYXNzIGxldCBhIGNsZWFuIGRvbWluYW50IHN1Yi1jZW50ZXIgZm9ybSB3aGlsZSBub2lzeQp3b29kYmxvY2sgdmFyaWFudHMgLyBtaXMtY3V0IGNyb3BzIGxhbmQgb24gdGhlIE9USEVSIHN1Yi1jZW50ZXJzIOKAlCBkaXJlY3RseQp0YXJnZXRpbmcgdGhlIG5vaXN5LWF1dG8tbGFiZWwgb3JpZ2luIG9mIHRoZSBjdXJyZW50IDAuNTcgZXJyb3ItQVVDIChyb2FkbWFwIFAyKS4KCkF0IGV4cG9ydCB0aW1lIChleHBvcnRfY2hlY2twb2ludC5weSkgdGhlIEsgc3ViLWNlbnRlcnMgYXJlIGNvbGxhcHNlZCB0byBPTkUKdmVjdG9yIHBlciBjbGFzcyDihpIgaGVhZFsiVyJdIG9mIHNoYXBlIChuX2NsYXNzZXMsIGVtYmVkX2RpbSksIHdoaWNoIGlzIHdoYXQKaW5mZXIucHkgbXVsdGlwbGllcyBmb3IgdGhlIE1heC1Mb2dpdCAvIGhlYWQtbG9naXQgZ2F0ZSAocm9hZG1hcCBQMCkuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKaW1wb3J0IHRvcmNodmlzaW9uCgpfQkFDS0JPTkVTID0gewogICAgInJlc25ldDE4IjogKHRvcmNodmlzaW9uLm1vZGVscy5yZXNuZXQxOCwgIlJlc05ldDE4X1dlaWdodHMiKSwKICAgICJyZXNuZXQzNCI6ICh0b3JjaHZpc2lvbi5tb2RlbHMucmVzbmV0MzQsICJSZXNOZXQzNF9XZWlnaHRzIiksCiAgICAicmVzbmV0NTAiOiAodG9yY2h2aXNpb24ubW9kZWxzLnJlc25ldDUwLCAiUmVzTmV0NTBfV2VpZ2h0cyIpLAp9CgoKY2xhc3MgTm9tRW1iZWRkZXIobm4uTW9kdWxlKToKICAgICIiIkltYWdlIC0+IEwyLW5vcm1hbGl6ZWQgZW1iZWRkaW5nLiBJREVOVElDQUwgc3RhdGVfZGljdCBsYXlvdXQgdG8gdGhlIHJlcG8KICAgIChiYWNrYm9uZS4qICsgcHJvai4qKSBzbyB0aGUgZXhwb3J0ZWQgYGJhY2tib25lYCBsb2FkcyBpbnRvIE5vbUVuY29kZXIuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkX2RpbTogaW50ID0gMjU2LCBwcmV0cmFpbmVkOiBib29sID0gVHJ1ZSwgYXJjaDogc3RyID0gInJlc25ldDE4Iik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgY3Rvciwgd25hbWUgPSBfQkFDS0JPTkVTW2FyY2hdCiAgICAgICAgd2VpZ2h0cyA9IGdldGF0dHIodG9yY2h2aXNpb24ubW9kZWxzLCB3bmFtZSkuSU1BR0VORVQxS19WMSBpZiBwcmV0cmFpbmVkIGVsc2UgTm9uZQogICAgICAgIGJiID0gY3Rvcih3ZWlnaHRzPXdlaWdodHMpCiAgICAgICAgaW5fZmVhdHMgPSBiYi5mYy5pbl9mZWF0dXJlcwogICAgICAgIGJiLmZjID0gbm4uSWRlbnRpdHkoKQogICAgICAgIHNlbGYuYXJjaCA9IGFyY2gKICAgICAgICBzZWxmLmJhY2tib25lID0gYmIKICAgICAgICBzZWxmLnByb2ogPSBubi5MaW5lYXIoaW5fZmVhdHMsIGVtYmVkX2RpbSkKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICByZXR1cm4gRi5ub3JtYWxpemUoc2VsZi5wcm9qKHNlbGYuYmFja2JvbmUoeCkpLCBkaW09MSkKCgpjbGFzcyBTdWJDZW50ZXJBcmNNYXJnaW4obm4uTW9kdWxlKToKICAgICIiIlN1Yi1jZW50ZXIgQXJjRmFjZSBoZWFkICh0cmFpbmluZyBvbmx5KS4KCiAgICBXOiAobl9jbGFzc2VzICogSywgZW1iZWRfZGltKS4gRm9yIGVhY2ggc2FtcGxlIHdlIHRha2UgdGhlIE1BWCBjb3NpbmUgb3ZlciB0aGUKICAgIEsgc3ViLWNlbnRlcnMgb2YgZWFjaCBjbGFzcyAodGhlIHNhbXBsZSBpcyBqdWRnZWQgYnkgaXRzIG5lYXJlc3Qgc3ViLWNlbnRlciBvZgogICAgYSBjbGFzcyksIHRoZW4gYXBwbHkgdGhlIGFkZGl0aXZlIGFuZ3VsYXIgbWFyZ2luIG9uIHRoZSB0YXJnZXQgY2xhc3Mgb25seS4KCiAgICBLPTEgcmVkdWNlcyB0byB2YW5pbGxhIEFyY0ZhY2UuIEs9MyBpcyB0aGUgRUNDVicyMCBkZWZhdWx0IGZvciBub2lzeSBsYWJlbHMuCiAgICAiIiIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgZW1iZWRfZGltOiBpbnQsIG5fY2xhc3NlczogaW50LCBrOiBpbnQgPSAzLAogICAgICAgICAgICAgICAgIHM6IGZsb2F0ID0gMzAuMCwgbTogZmxvYXQgPSAwLjMwKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLm5fY2xhc3Nlcywgc2VsZi5rLCBzZWxmLnMsIHNlbGYubSA9IG5fY2xhc3NlcywgaywgcywgbQogICAgICAgIHNlbGYuVyA9IG5uLlBhcmFtZXRlcih0b3JjaC5yYW5kbihuX2NsYXNzZXMgKiBrLCBlbWJlZF9kaW0pKQogICAgICAgIG5uLmluaXQueGF2aWVyX3VuaWZvcm1fKHNlbGYuVykKCiAgICBkZWYgc3ViX2Nvc2luZXMoc2VsZiwgZW1iKToKICAgICAgICAiIiIoQiwgbl9jbGFzc2VzKSBtYXgtb3Zlci1zdWJjZW50ZXIgY29zaW5lLiBlbWIgaXMgYWxyZWFkeSBMMi1ub3JtYWxpemVkLiIiIgogICAgICAgIGNvcyA9IGVtYiBAIEYubm9ybWFsaXplKHNlbGYuVywgZGltPTEpLnQoKSAgICAgICAgICAgICAgICMgKEIsIG5fY2xzKkspCiAgICAgICAgY29zID0gY29zLnZpZXcoLTEsIHNlbGYubl9jbGFzc2VzLCBzZWxmLmspICAgICAgICAgICAgICAgIyAoQiwgbl9jbHMsIEspCiAgICAgICAgcmV0dXJuIGNvcy5tYXgoZGltPTIpLnZhbHVlcyAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyAoQiwgbl9jbHMpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgZW1iLCBsYWJlbHM9Tm9uZSk6CiAgICAgICAgY29zID0gc2VsZi5zdWJfY29zaW5lcyhlbWIpCiAgICAgICAgaWYgbGFiZWxzIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBjb3MgKiBzZWxmLnMKICAgICAgICBjb3MgPSBjb3MuY2xhbXAoLTEgKyAxZS02LCAxIC0gMWUtNikKICAgICAgICB0YXJnZXQgPSB0b3JjaC5jb3ModG9yY2guYWNvcyhjb3MpICsgc2VsZi5tKQogICAgICAgIG9uZWhvdCA9IEYub25lX2hvdChsYWJlbHMsIHNlbGYubl9jbGFzc2VzKS50byhjb3MuZHR5cGUpCiAgICAgICAgcmV0dXJuIChvbmVob3QgKiB0YXJnZXQgKyAoMSAtIG9uZWhvdCkgKiBjb3MpICogc2VsZi5zCg==",
    "dataset.py": "IiIiRGF0YXNldCArIHNwbGl0cyArIHNhbXBsZXJzLgoKRml4ZXMgdGhlIHRocmVlIGRhdGEtc2lkZSBkZWZlY3RzIHRoZSBhdWRpdCBtZWFzdXJlZCBvbiB0aGUgT0xEIGVuY29kZXI6CiAgKiBQMiBsZWFrYWdlIOKAlCB0aGUgb2xkIHNwbGl0IHdhcyBSQU5ET00gY3JvcC1sZXZlbCwgc28gODYlIG9mIHBhZ2VzIGhhZCBjcm9wcwogICAgaW4gPjEgc3BsaXQg4oaSIHZhbC90ZXN0IHdlcmUgY29udGFtaW5hdGVkIGJ5IHNhbWUtc2NhbiBuZWlnaGJvdXJzLgogICAgYGFzc2lnbl9zcGxpdHMobW9kZT0icGFnZV9kaXNqb2ludCIpYCBzcGxpdHMgYnkgKGJvb2sscGFnZSk7IG1vZGU9ImxvYm8iCiAgICBob2xkcyBvdXQgYSB3aG9sZSBib29rIChsZWF2ZS1vbmUtYm9vay1vdXQpIGZvciBhbiBob25lc3QgY3Jvc3MtYm9vayBudW1iZXIuCiAgKiBsb25nIHRhaWwg4oCUIDUyMiBjbGFzc2VzIGhhdmUgMSBjcm9wLCAxMDE3IGhhdmUgPDguIGBjbGFzc19iYWxhbmNlZF93ZWlnaHRzYAogICAgZmVlZHMgYSBXZWlnaHRlZFJhbmRvbVNhbXBsZXIgc28gcmFyZSBnbHlwaHMgYXJlIHNlZW4gYXMgb2Z0ZW4gYXMg6bq7ICgxNjQ0KS4KICAqIGNvbmZ1c2FibGUgcGFpcnMgKOOdtS9uZ8aw4budaS10eXBlKSDigJQgYENvbmZ1c2lvbkJhdGNoU2FtcGxlcmAgY28tbG9jYXRlcyBhIGNsYXNzCiAgICB3aXRoIGl0cyBTaW5vTm9tLXNpbWlsYXIgY2hhcnMgaW4gdGhlIHNhbWUgYmF0Y2ggc28gQXJjRmFjZSdzIG1hcmdpbiBpcyBmb3JjZWQKICAgIHRvIHNlcGFyYXRlIHRoZSBleGFjdCBsb29rYWxpa2VzIHRoYXQgZHJpdmUgdGhlIGVycm9ycyAoaGFyZC1uZWdhdGl2ZSBtaW5pbmcpLgoKSW1hZ2UgZnJhbWluZyBpcyBieXRlLWlkZW50aWNhbCB0byBpbmZlci5Ob21FbmNvZGVyLl9wcmVwIChzcXVhcmUgd2hpdGUtcGFkIOKGkgpyZXNpemUg4oaSIC8yNTUg4oaSIG1lYW4vc3RkIDAuNSwgM2NoKSBzbyB0cmFpbiBhbmQgaW5mZXJlbmNlIHNlZSB0aGUgc2FtZSBwaXhlbHMuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaGFzaGxpYgppbXBvcnQgcmFuZG9tCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CgppbXBvcnQgY3YyCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhc2V0LCBTYW1wbGVyCgpNRUFOID0gbnAuYXJyYXkoWzAuNSwgMC41LCAwLjVdLCBucC5mbG9hdDMyKQpTVEQgPSBucC5hcnJheShbMC41LCAwLjUsIDAuNV0sIG5wLmZsb2F0MzIpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIFNwbGl0cyDigJQgcGFnZS1kaXNqb2ludCAoZGVmYXVsdCkgb3IgbGVhdmUtb25lLWJvb2stb3V0CiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tICMKZGVmIGFzc2lnbl9zcGxpdHMoZGYsIG1vZGU9InBhZ2VfZGlzam9pbnQiLCBob2xkb3V0X2Jvb2s9IiIsIHZhbF9mcmFjPTAuMSwKICAgICAgICAgICAgICAgICAgdGVzdF9mcmFjPTAuMSwgc2VlZD00Mik6CiAgICAiIiJSZXR1cm4gYSBsaXN0IG9mICd0cmFpbicvJ3ZhbCcvJ3Rlc3QnIGFsaWduZWQgdG8gZGYgcm93cy4KCiAgICBwYWdlX2Rpc2pvaW50OiBoYXNoIChib29rLHBhZ2UpIOKGkiBhIHdob2xlIHBhZ2UgbGFuZHMgZW50aXJlbHkgaW4gb25lIHNwbGl0LCBzbwogICAgICBubyBjcm9wIG9mIGEgdHJhaW4gcGFnZSBjYW4gYXBwZWFyIGluIHZhbC90ZXN0LgogICAgbG9ibzogZXZlcnkgY3JvcCBvZiBgaG9sZG91dF9ib29rYCDihpIgdGVzdDsgdGhlIHJlc3Qgc3BsaXQgcGFnZS1kaXNqb2ludCBpbnRvCiAgICAgIHRyYWluL3ZhbCDigJQgdGhlIGhvbmVzdCAiZG9lcyBpdCBnZW5lcmFsaXNlIHRvIGFuIHVuc2VlbiBib29rIiBzZXR0aW5nLgogICAgRkQgc3ludGhldGljIGdseXBocyAoc291cmNlPT0nZmQnKSBBTFdBWVMgZ28gdG8gdHJhaW4gKHRoZXkgYXJlIHJlZmVyZW5jZXMsCiAgICBuZXZlciBhbiBldmFsdWF0aW9uIHRhcmdldCkuCiAgICAiIiIKICAgIG91dCA9IFtdCiAgICBmb3IgXywgciBpbiBkZi5pdGVycm93cygpOgogICAgICAgIGlmIHN0cihyLmdldCgic291cmNlIikpID09ICJmZCI6CiAgICAgICAgICAgIG91dC5hcHBlbmQoInRyYWluIik7IGNvbnRpbnVlCiAgICAgICAgYm9vaywgcGFnZSA9IHN0cihyWyJib29rIl0pLCBzdHIoclsicGFnZSJdKQogICAgICAgIGlmIG1vZGUgPT0gImxvYm8iIGFuZCBob2xkb3V0X2Jvb2sgYW5kIGJvb2sgPT0gaG9sZG91dF9ib29rOgogICAgICAgICAgICBvdXQuYXBwZW5kKCJ0ZXN0Iik7IGNvbnRpbnVlCiAgICAgICAgaCA9IGludChoYXNobGliLm1kNShmIntzZWVkfXx7Ym9va318e3BhZ2V9Ii5lbmNvZGUoKSkuaGV4ZGlnZXN0KCksIDE2KSAlIDEwMDAgLyAxMDAwLjAKICAgICAgICBpZiBtb2RlID09ICJsb2JvIjoKICAgICAgICAgICAgb3V0LmFwcGVuZCgidmFsIiBpZiBoIDwgdmFsX2ZyYWMgLyAoMSAtIHRlc3RfZnJhYykgZWxzZSAidHJhaW4iKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIG91dC5hcHBlbmQoInRlc3QiIGlmIGggPCB0ZXN0X2ZyYWMgZWxzZSAoInZhbCIgaWYgaCA8IHRlc3RfZnJhYyArIHZhbF9mcmFjIGVsc2UgInRyYWluIikpCiAgICByZXR1cm4gb3V0CgoKZGVmIGNsYXNzX2JhbGFuY2VkX3dlaWdodHMobGFiZWxzX2lkeCwgbl9jbGFzc2VzKToKICAgICIiIkludmVyc2UtZnJlcXVlbmN5IHdlaWdodCBwZXIgc2FtcGxlIGZvciBXZWlnaHRlZFJhbmRvbVNhbXBsZXIgKGxvbmcgdGFpbCkuIiIiCiAgICBmcmVxID0gbnAuYmluY291bnQobGFiZWxzX2lkeCwgbWlubGVuZ3RoPW5fY2xhc3NlcykuYXN0eXBlKG5wLmZsb2F0NjQpCiAgICBmcmVxW2ZyZXEgPT0gMF0gPSAxLjAKICAgIHcgPSAxLjAgLyBmcmVxW2xhYmVsc19pZHhdCiAgICByZXR1cm4gdG9yY2guYXNfdGVuc29yKHcsIGR0eXBlPXRvcmNoLmRvdWJsZSkKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCiMgQ29uZnVzaW9uLWF3YXJlIGJhdGNoZXMgKGhhcmQtbmVnYXRpdmUgbWluaW5nKQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSAjCmNsYXNzIENvbmZ1c2lvbkJhdGNoU2FtcGxlcihTYW1wbGVyKToKICAgICIiIkJ1aWxkIGJhdGNoZXMgdGhhdCBkZWxpYmVyYXRlbHkgbWl4IGEgY2xhc3Mgd2l0aCBpdHMgdmlzdWFsIGxvb2thbGlrZXMuCgogICAgc2ltaWxhcl9tYXA6IHtjbGFzc19pZHg6IFtzaW1pbGFyX2NsYXNzX2lkeCwgLi4uXX0gYnVpbHQgZnJvbSB0aGUgU2lub05vbQogICAgc2ltaWxhcml0eSBkaWN0IChvbmx5IHBhaXJzIHdob3NlIEJPVEggY2hhcnMgZXhpc3QgaW4gdGhpcyB0cmFpbmluZyBzZXQpLgogICAgRWFjaCBiYXRjaDogcGljayBhIHNlZWQgY2xhc3MsIGFkZCBpdHMgcHJlc2VudCBzaW1pbGFycywgdGhlbiBmaWxsIHdpdGggcmFuZG9tCiAgICBjbGFzc2VzOyBkcmF3IGBwZXJfY2xhc3NgIGNyb3BzIGZvciBldmVyeSBjaG9zZW4gY2xhc3MuIEZvcmNlcyB0aGUgbWFyZ2luIHRvCiAgICBzZXBhcmF0ZSB0aGUgZXhhY3QgY29uZnVzaW9ucyB0aGF0IHByb2R1Y2Ugd3JvbmcgbGFiZWxzLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGxhYmVsc19pZHgsIHNpbWlsYXJfbWFwLCBiYXRjaF9zaXplPTEyOCwgcGVyX2NsYXNzPTQsCiAgICAgICAgICAgICAgICAgc2VlZD00MiwgbGVuZ3RoPU5vbmUpOgogICAgICAgIHNlbGYuYnlfY2xhc3MgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgICAgIGZvciBpLCBjIGluIGVudW1lcmF0ZShsYWJlbHNfaWR4KToKICAgICAgICAgICAgc2VsZi5ieV9jbGFzc1tpbnQoYyldLmFwcGVuZChpKQogICAgICAgIHNlbGYuY2xhc3NlcyA9IFtjIGZvciBjLCB2IGluIHNlbGYuYnlfY2xhc3MuaXRlbXMoKSBpZiB2XQogICAgICAgIHNlbGYuc2ltaWxhcl9tYXAgPSBzaW1pbGFyX21hcAogICAgICAgIHNlbGYuYmF0Y2hfc2l6ZSA9IGJhdGNoX3NpemUKICAgICAgICBzZWxmLnBlcl9jbGFzcyA9IG1heCgxLCBwZXJfY2xhc3MpCiAgICAgICAgc2VsZi5ybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICAgICAgc2VsZi5sZW5ndGggPSBsZW5ndGggb3IgKGxlbihsYWJlbHNfaWR4KSAvLyBiYXRjaF9zaXplKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiBzZWxmLmxlbmd0aAoKICAgIGRlZiBfX2l0ZXJfXyhzZWxmKToKICAgICAgICBmb3IgXyBpbiByYW5nZShzZWxmLmxlbmd0aCk6CiAgICAgICAgICAgIGNob3Nlbiwgc2VlbiA9IFtdLCBzZXQoKQogICAgICAgICAgICB3aGlsZSBsZW4oY2hvc2VuKSAqIHNlbGYucGVyX2NsYXNzIDwgc2VsZi5iYXRjaF9zaXplOgogICAgICAgICAgICAgICAgc2VlZF9jID0gc2VsZi5ybmcuY2hvaWNlKHNlbGYuY2xhc3NlcykKICAgICAgICAgICAgICAgIGdyb3VwID0gW3NlZWRfY10gKyBbcyBmb3IgcyBpbiBzZWxmLnNpbWlsYXJfbWFwLmdldChzZWVkX2MsIFtdKSBpZiBzIGluIHNlbGYuYnlfY2xhc3NdCiAgICAgICAgICAgICAgICBmb3IgYyBpbiBncm91cDoKICAgICAgICAgICAgICAgICAgICBpZiBjIGluIHNlZW46CiAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgc2Vlbi5hZGQoYyk7IGNob3Nlbi5hcHBlbmQoYykKICAgICAgICAgICAgICAgICAgICBpZiBsZW4oY2hvc2VuKSAqIHNlbGYucGVyX2NsYXNzID49IHNlbGYuYmF0Y2hfc2l6ZToKICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgYmF0Y2ggPSBbXQogICAgICAgICAgICBmb3IgYyBpbiBjaG9zZW46CiAgICAgICAgICAgICAgICBwb29sID0gc2VsZi5ieV9jbGFzc1tjXQogICAgICAgICAgICAgICAgYmF0Y2ggKz0gKHNlbGYucm5nLmNob2ljZXMocG9vbCwgaz1zZWxmLnBlcl9jbGFzcykgaWYgbGVuKHBvb2wpIDwgc2VsZi5wZXJfY2xhc3MKICAgICAgICAgICAgICAgICAgICAgICAgICBlbHNlIHNlbGYucm5nLnNhbXBsZShwb29sLCBzZWxmLnBlcl9jbGFzcykpCiAgICAgICAgICAgIHlpZWxkIGJhdGNoWzpzZWxmLmJhdGNoX3NpemVdCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwojIERhdGFzZXQKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0gIwpjbGFzcyBOb21Dcm9wRGF0YXNldChEYXRhc2V0KToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXRocywgbGFiZWxzX2lkeCwgaW1nPTEyOCwgdHJhaW49VHJ1ZSwgYXVnPVRydWUsIHdlaWdodHM9Tm9uZSk6CiAgICAgICAgc2VsZi5wYXRocyA9IGxpc3QocGF0aHMpCiAgICAgICAgc2VsZi55ID0gbGlzdChsYWJlbHNfaWR4KQogICAgICAgIHNlbGYuaW1nID0gaW1nCiAgICAgICAgc2VsZi50cmFpbiA9IHRyYWluIGFuZCBhdWcKICAgICAgICAjIHBlci1zYW1wbGUgbG9zcyB3ZWlnaHQgKHRpZXI6IEdPTEQgMS4wID4gU0lMVkVSIDAuNSA+IEZEIDAuNCkg4oCUIGxldHMgdGhlCiAgICAgICAgIyB0cmFpbmVyIHRydXN0IGNsZWFuIEdPTEQgbW9yZSB0aGFuIG5vaXN5IEFJLWF1ZGl0ZWQgU0lMVkVSIChkZS1jaXJjdWxhcikuCiAgICAgICAgc2VsZi53ID0gbGlzdCh3ZWlnaHRzKSBpZiB3ZWlnaHRzIGlzIG5vdCBOb25lIGVsc2UgWzEuMF0gKiBsZW4oc2VsZi5wYXRocykKCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gbGVuKHNlbGYucGF0aHMpCgogICAgZGVmIF9zcXVhcmUoc2VsZiwgZyk6CiAgICAgICAgaCwgdyA9IGcuc2hhcGUKICAgICAgICBzID0gbWF4KGgsIHcpCiAgICAgICAgY2FudmFzID0gbnAuZnVsbCgocywgcyksIDI1NSwgbnAudWludDgpCiAgICAgICAgY2FudmFzWyhzIC0gaCkgLy8gMjoocyAtIGgpIC8vIDIgKyBoLCAocyAtIHcpIC8vIDI6KHMgLSB3KSAvLyAyICsgd10gPSBnCiAgICAgICAgcmV0dXJuIGN2Mi5yZXNpemUoY2FudmFzLCAoc2VsZi5pbWcsIHNlbGYuaW1nKSwgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfQVJFQSkKCiAgICBkZWYgX2F1Z21lbnQoc2VsZiwgZyk6CiAgICAgICAgIyBnZW9tZXRyeSBvbmx5IOKAlCBOTyBob3Jpem9udGFsIGZsaXAgKGdseXBocyBhcmUgbm90IG1pcnJvci1pbnZhcmlhbnQpCiAgICAgICAgYSA9IHNlbGYuaW1nCiAgICAgICAgYW5nID0gbnAucmFuZG9tLnVuaWZvcm0oLTgsIDgpCiAgICAgICAgdHgsIHR5ID0gbnAucmFuZG9tLnVuaWZvcm0oLTAuMDYsIDAuMDYsIDIpICogYQogICAgICAgIHNjID0gbnAucmFuZG9tLnVuaWZvcm0oMC45MCwgMS4xMCkKICAgICAgICBNID0gY3YyLmdldFJvdGF0aW9uTWF0cml4MkQoKGEgLyAyLCBhIC8gMiksIGFuZywgc2MpCiAgICAgICAgTVs6LCAyXSArPSAodHgsIHR5KQogICAgICAgIGcgPSBjdjIud2FycEFmZmluZShnLCBNLCAoYSwgYSksIGJvcmRlclZhbHVlPTI1NSwgZmxhZ3M9Y3YyLklOVEVSX0xJTkVBUikKICAgICAgICBpZiBucC5yYW5kb20ucmFuZCgpIDwgMC4zOiAgICAgICAgICAgICAgICAgICAgICAgIyBpbmsgdGhpY2tuZXNzIGppdHRlcgogICAgICAgICAgICBrID0gbnAub25lcygoMiwgMiksIG5wLnVpbnQ4KQogICAgICAgICAgICBnID0gY3YyLmVyb2RlKGcsIGspIGlmIG5wLnJhbmRvbS5yYW5kKCkgPCAwLjUgZWxzZSBjdjIuZGlsYXRlKGcsIGspCiAgICAgICAgaWYgbnAucmFuZG9tLnJhbmQoKSA8IDAuMjU6ICAgICAgICAgICAgICAgICAgICAgICMgcmFuZG9tLWVyYXNpbmcgKG9jY2x1c2lvbikKICAgICAgICAgICAgZWgsIGV3ID0gbnAucmFuZG9tLnJhbmRpbnQoYSAvLyA4LCBhIC8vIDMsIDIpCiAgICAgICAgICAgIHkwLCB4MCA9IG5wLnJhbmRvbS5yYW5kaW50KDAsIGEgLSBlaCksIG5wLnJhbmRvbS5yYW5kaW50KDAsIGEgLSBldykKICAgICAgICAgICAgZ1t5MDp5MCArIGVoLCB4MDp4MCArIGV3XSA9IDI1NQogICAgICAgIHJldHVybiBnCgogICAgZGVmIF9fZ2V0aXRlbV9fKHNlbGYsIGkpOgogICAgICAgIGcgPSBjdjIuaW1yZWFkKHN0cihzZWxmLnBhdGhzW2ldKSwgY3YyLklNUkVBRF9HUkFZU0NBTEUpCiAgICAgICAgaWYgZyBpcyBOb25lOgogICAgICAgICAgICBnID0gbnAuZnVsbCgoc2VsZi5pbWcsIHNlbGYuaW1nKSwgMjU1LCBucC51aW50OCkKICAgICAgICBlbHNlOgogICAgICAgICAgICBnID0gc2VsZi5fc3F1YXJlKGcpCiAgICAgICAgaWYgc2VsZi50cmFpbjoKICAgICAgICAgICAgZyA9IHNlbGYuX2F1Z21lbnQoZykKICAgICAgICB4ID0gbnAucmVwZWF0KChnW05vbmVdLmFzdHlwZShucC5mbG9hdDMyKSAvIDI1NS4wKSwgMywgYXhpcz0wKQogICAgICAgIHggPSAoeCAtIE1FQU5bOiwgTm9uZSwgTm9uZV0pIC8gU1REWzosIE5vbmUsIE5vbmVdCiAgICAgICAgcmV0dXJuIHRvcmNoLmZyb21fbnVtcHkoeCksIGludChzZWxmLnlbaV0pLCBmbG9hdChzZWxmLndbaV0pCg==",
    "sam.py": "IiIiU0FNIOKAlCBTaGFycG5lc3MtQXdhcmUgTWluaW1pemF0aW9uIChGb3JldCBldCBhbC4sIElDTFIgMjAyMSkuCgpGbGF0LW1pbmltYSB0cmFpbmluZyBpcyB0aGUgcm9hZG1hcC1QMiBsZXZlciBmb3IgRkFJTFVSRSBQUkVESUNUSU9OOiBaaHUgZXQgYWwuCihFQ0NWIDIwMjIsICJSZXRoaW5raW5nIENvbmZpZGVuY2UgQ2FsaWJyYXRpb24gZm9yIEZhaWx1cmUgUHJlZGljdGlvbiIpIHNob3cgdGhhdApvcmRpbmFyeSB0cmFpbmluZy9jYWxpYnJhdGlvbiBIVVJUUyB0aGUgYWJpbGl0eSB0byB0ZWxsIGNvcnJlY3QgZnJvbSB3cm9uZwpwcmVkaWN0aW9ucywgd2hpbGUgZmxhdCBtaW5pbWEgKFNBTS9TV0EpIHdpZGVuIHRoZSBjb3JyZWN0LXZzLXdyb25nIGNvbmZpZGVuY2UKZ2FwIOKAlCBpLmUuIHRoZXkgcmFpc2UgdGhlIGVycm9yLWRldGVjdGlvbiBBVUMgaXRzZWxmLCB3aGljaCBpcyBleGFjdGx5IHRoZSBtZXRyaWMKdGhlIGN1cnJlbnQgZW5jb2RlciBmYWlscyAoMC41NykuIFNBTSB3cmFwcyBhbnkgYmFzZSBvcHRpbWl6ZXI7IHVzZSB3aXRoIHRoZQp0d28tc3RlcCBjbG9zdXJlIGluIHRyYWluLnB5LiBTV0EgaXMgYXBwbGllZCBzZXBhcmF0ZWx5IGluIHRyYWluLnB5LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRvcmNoCgoKY2xhc3MgU0FNKHRvcmNoLm9wdGltLk9wdGltaXplcik6CiAgICBkZWYgX19pbml0X18oc2VsZiwgcGFyYW1zLCBiYXNlX29wdGltaXplciwgcmhvOiBmbG9hdCA9IDAuMDUsIGFkYXB0aXZlOiBib29sID0gRmFsc2UsICoqa3cpOgogICAgICAgIGFzc2VydCByaG8gPj0gMAogICAgICAgIGRlZmF1bHRzID0gZGljdChyaG89cmhvLCBhZGFwdGl2ZT1hZGFwdGl2ZSwgKiprdykKICAgICAgICBzdXBlcigpLl9faW5pdF9fKHBhcmFtcywgZGVmYXVsdHMpCiAgICAgICAgc2VsZi5iYXNlX29wdGltaXplciA9IGJhc2Vfb3B0aW1pemVyKHNlbGYucGFyYW1fZ3JvdXBzLCAqKmt3KQogICAgICAgIHNlbGYucGFyYW1fZ3JvdXBzID0gc2VsZi5iYXNlX29wdGltaXplci5wYXJhbV9ncm91cHMKICAgICAgICBzZWxmLmRlZmF1bHRzLnVwZGF0ZShzZWxmLmJhc2Vfb3B0aW1pemVyLmRlZmF1bHRzKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBmaXJzdF9zdGVwKHNlbGYsIHplcm9fZ3JhZD1GYWxzZSk6CiAgICAgICAgZ3JhZF9ub3JtID0gc2VsZi5fZ3JhZF9ub3JtKCkKICAgICAgICBmb3IgZ3JvdXAgaW4gc2VsZi5wYXJhbV9ncm91cHM6CiAgICAgICAgICAgIHNjYWxlID0gZ3JvdXBbInJobyJdIC8gKGdyYWRfbm9ybSArIDFlLTEyKQogICAgICAgICAgICBmb3IgcCBpbiBncm91cFsicGFyYW1zIl06CiAgICAgICAgICAgICAgICBpZiBwLmdyYWQgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgc2VsZi5zdGF0ZVtwXVsib2xkX3AiXSA9IHAuZGF0YS5jbG9uZSgpCiAgICAgICAgICAgICAgICBlX3cgPSAodG9yY2gucG93KHAsIDIpIGlmIGdyb3VwWyJhZGFwdGl2ZSJdIGVsc2UgMS4wKSAqIHAuZ3JhZCAqIHNjYWxlLnRvKHApCiAgICAgICAgICAgICAgICBwLmFkZF8oZV93KSAgICAgICAgICAgICAgICAgICAgICAgICAjIGNsaW1iIHRvIHRoZSBsb2NhbCB3b3JzdCBwb2ludAogICAgICAgIGlmIHplcm9fZ3JhZDoKICAgICAgICAgICAgc2VsZi56ZXJvX2dyYWQoKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBzZWNvbmRfc3RlcChzZWxmLCB6ZXJvX2dyYWQ9RmFsc2UpOgogICAgICAgIGZvciBncm91cCBpbiBzZWxmLnBhcmFtX2dyb3VwczoKICAgICAgICAgICAgZm9yIHAgaW4gZ3JvdXBbInBhcmFtcyJdOgogICAgICAgICAgICAgICAgaWYgcC5ncmFkIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHAuZGF0YSA9IHNlbGYuc3RhdGVbcF1bIm9sZF9wIl0gICAgICMgZ28gYmFjayB0byB0aGUgb3JpZ2luYWwgd2VpZ2h0cwogICAgICAgIHNlbGYuYmFzZV9vcHRpbWl6ZXIuc3RlcCgpICAgICAgICAgICAgICAgICAgIyBkbyB0aGUgYWN0dWFsIHVwZGF0ZQogICAgICAgIGlmIHplcm9fZ3JhZDoKICAgICAgICAgICAgc2VsZi56ZXJvX2dyYWQoKQoKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBfZ3JhZF9ub3JtKHNlbGYpOgogICAgICAgIHNoYXJlZF9kZXZpY2UgPSBzZWxmLnBhcmFtX2dyb3Vwc1swXVsicGFyYW1zIl1bMF0uZGV2aWNlCiAgICAgICAgbm9ybXMgPSBbXQogICAgICAgIGZvciBncm91cCBpbiBzZWxmLnBhcmFtX2dyb3VwczoKICAgICAgICAgICAgZm9yIHAgaW4gZ3JvdXBbInBhcmFtcyJdOgogICAgICAgICAgICAgICAgaWYgcC5ncmFkIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIGcgPSAodG9yY2guYWJzKHApIGlmIGdyb3VwWyJhZGFwdGl2ZSJdIGVsc2UgMS4wKSAqIHAuZ3JhZAogICAgICAgICAgICAgICAgbm9ybXMuYXBwZW5kKGcubm9ybShwPTIpLnRvKHNoYXJlZF9kZXZpY2UpKQogICAgICAgIHJldHVybiB0b3JjaC5ub3JtKHRvcmNoLnN0YWNrKG5vcm1zKSwgcD0yKQoKICAgIGRlZiBzdGVwKHNlbGYsIGNsb3N1cmU9Tm9uZSk6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJTQU0gcmVxdWlyZXMgdGhlIHR3by1zdGVwIGZvcm06IGZpcnN0X3N0ZXAoKS9zZWNvbmRfc3RlcCgpIGluIHRyYWluLnB5IikK",
    "hub.py": "IiIiSHVnZ2luZyBGYWNlIEh1YiBzeW5jIOKAlCBzdXJ2aXZlIEthZ2dsZSBzZXNzaW9uIHJlc2V0cy4KCkthZ2dsZSBHUFUgc2Vzc2lvbnMgZGllICg5aCBjYXAsIGludGVycnVwdHMsICJyZXNldCIpLiBXaXRob3V0IHRoaXMsIGEgcmVzZXQgbG9zZXMKYWxsIHRyYWluaW5nLiBXaXRoIGl0LCB0cmFpbi5weSBwdXNoZXMgYSBGVUxMLVNUQVRFIGBsYXN0LnB0YCAobW9kZWwgKyBoZWFkICsKb3B0aW1pemVyICsgc2NoZWR1bGVyICsgZXBvY2ggKyBybmcpIHRvIGEgcHJpdmF0ZSBIRiByZXBvIGV2ZXJ5IGVwb2NoLCBhbmQgb24KcmVzdGFydCBwdWxscyBpdCBiYWNrIGFuZCBDT05USU5VRVMgZnJvbSB0aGUgc2F2ZWQgZXBvY2ggaW5zdGVhZCBvZiBlcG9jaCAxLgoKRXZlcnl0aGluZyBkZWdyYWRlcyBncmFjZWZ1bGx5OiBubyBgaHVnZ2luZ2ZhY2VfaHViYCwgbm8gdG9rZW4sIG9yIG5vIGludGVybmV0IOKGkgpmdW5jdGlvbnMgbm8tb3AgLyByZXR1cm4gTm9uZSBhbmQgdHJhaW5pbmcganVzdCBydW5zIGxvY2FsbHkgKEthZ2dsZSAiU2F2ZQpWZXJzaW9uIiBvZiAva2FnZ2xlL3dvcmtpbmcgaXMgdGhlbiB5b3VyIG9ubHkgcGVyc2lzdGVuY2UpLiBUaGUgcmVwbydzIGV4aXN0aW5nCndlaWdodHMgbGl2ZSBhdCBIRiBgbWRudDU3MS9ub20tZW1iZWRgLCBzbyB0aGUgc2FtZSBhY2NvdW50L3Rva2VuIHdvcmtzIGhlcmUuCgpUb2tlbjogcGFzcyAtLWhmLXRva2VuLCBvciBzZXQgZW52IEhGX1RPS0VOLCBvciBvbiBLYWdnbGUgYWRkIGEgU2VjcmV0IG5hbWVkCkhGX1RPS0VOIChBZGQtb25zIOKGkiBTZWNyZXRzKSBhbmQgYG9zLmVudmlyb25gIHBpY2tzIGl0IHVwLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKCmRlZiBfaHViKCk6CiAgICB0cnk6CiAgICAgICAgaW1wb3J0IGh1Z2dpbmdmYWNlX2h1YiAgIyBub3FhOiBGNDAxCiAgICAgICAgcmV0dXJuIGh1Z2dpbmdmYWNlX2h1YgogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiByZXNvbHZlX3Rva2VuKGNsaV90b2tlbjogc3RyID0gIiIpIC0+IHN0ciB8IE5vbmU6CiAgICBpZiBjbGlfdG9rZW46CiAgICAgICAgcmV0dXJuIGNsaV90b2tlbgogICAgZm9yIGsgaW4gKCJIRl9UT0tFTiIsICJIVUdHSU5HX0ZBQ0VfSFVCX1RPS0VOIiwgIkhVR0dJTkdGQUNFX1RPS0VOIik6CiAgICAgICAgaWYgb3MuZW52aXJvbi5nZXQoayk6CiAgICAgICAgICAgIHJldHVybiBvcy5lbnZpcm9uW2tdCiAgICAjIEthZ2dsZSBTZWNyZXRzIChpZiB0aGUgbm90ZWJvb2sgZW5hYmxlZCB0aGVtKQogICAgdHJ5OgogICAgICAgIGZyb20ga2FnZ2xlX3NlY3JldHMgaW1wb3J0IFVzZXJTZWNyZXRzQ2xpZW50CiAgICAgICAgcmV0dXJuIFVzZXJTZWNyZXRzQ2xpZW50KCkuZ2V0X3NlY3JldCgiSEZfVE9LRU4iKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiBlbnN1cmVfcmVwbyhyZXBvX2lkOiBzdHIsIHRva2VuOiBzdHIpIC0+IGJvb2w6CiAgICBodWIgPSBfaHViKCkKICAgIGlmIGh1YiBpcyBOb25lIG9yIG5vdCB0b2tlbiBvciBub3QgcmVwb19pZDoKICAgICAgICByZXR1cm4gRmFsc2UKICAgIHRyeToKICAgICAgICBodWIuY3JlYXRlX3JlcG8ocmVwb19pZCwgdG9rZW49dG9rZW4sIHByaXZhdGU9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSwgcmVwb190eXBlPSJtb2RlbCIpCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIltoZl0gY3JlYXRlX3JlcG8gc2tpcHBlZDoge2V9IikKICAgICAgICByZXR1cm4gRmFsc2UKCgpkZWYgcHVzaChsb2NhbF9wYXRoLCByZXBvX2lkOiBzdHIsIHBhdGhfaW5fcmVwbzogc3RyLCB0b2tlbjogc3RyKSAtPiBib29sOgogICAgIiIiVXBsb2FkIG9uZSBmaWxlLiBUcnVlIG9uIHN1Y2Nlc3MsIEZhbHNlIGlmIEhGIHVuYXZhaWxhYmxlL2ZhaWxlZCAobm9uLWZhdGFsKS4iIiIKICAgIGh1YiA9IF9odWIoKQogICAgaWYgaHViIGlzIE5vbmUgb3Igbm90IHRva2VuIG9yIG5vdCByZXBvX2lkIG9yIG5vdCBQYXRoKGxvY2FsX3BhdGgpLmV4aXN0cygpOgogICAgICAgIHJldHVybiBGYWxzZQogICAgdHJ5OgogICAgICAgIGh1Yi51cGxvYWRfZmlsZShwYXRoX29yX2ZpbGVvYmo9c3RyKGxvY2FsX3BhdGgpLCBwYXRoX2luX3JlcG89cGF0aF9pbl9yZXBvLAogICAgICAgICAgICAgICAgICAgICAgICByZXBvX2lkPXJlcG9faWQsIHRva2VuPXRva2VuLCByZXBvX3R5cGU9Im1vZGVsIikKICAgICAgICBwcmludChmIltoZl0gcHVzaGVkIHtwYXRoX2luX3JlcG99IC0+IHtyZXBvX2lkfSIpCiAgICAgICAgcmV0dXJuIFRydWUKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIltoZl0gcHVzaCBmYWlsZWQgKHtlfSkg4oCUIGNvbnRpbnVpbmcgbG9jYWwtb25seSIpCiAgICAgICAgcmV0dXJuIEZhbHNlCgoKZGVmIHB1bGwocmVwb19pZDogc3RyLCBmaWxlbmFtZTogc3RyLCB0b2tlbjogc3RyLCBkZXN0X2RpcikgLT4gc3RyIHwgTm9uZToKICAgICIiIkRvd25sb2FkIGEgZmlsZSBmcm9tIHRoZSByZXBvIOKGkiBsb2NhbCBwYXRoLCBvciBOb25lIGlmIGFic2VudC91bmF2YWlsYWJsZS4iIiIKICAgIGh1YiA9IF9odWIoKQogICAgaWYgaHViIGlzIE5vbmUgb3Igbm90IHJlcG9faWQ6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHRyeToKICAgICAgICByZXR1cm4gaHViLmhmX2h1Yl9kb3dubG9hZChyZXBvX2lkPXJlcG9faWQsIGZpbGVuYW1lPWZpbGVuYW1lLCB0b2tlbj10b2tlbiBvciBOb25lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHJlcG9fdHlwZT0ibW9kZWwiLCBsb2NhbF9kaXI9c3RyKGRlc3RfZGlyKSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIltoZl0gbm8gcmVzdW1lIGNoZWNrcG9pbnQgb24gaHViICh7ZX0pIikKICAgICAgICByZXR1cm4gTm9uZQo=",
    "train.py": "IiIiVHJhaW4gdGhlIE7DtG0gZW1iZWRkZXIgKyBTdWItY2VudGVyIEFyY0ZhY2UgaGVhZCAocm9hZG1hcCBQMikuCgpMZXZlcnMgd2lyZWQgaW46CiAgKiBwYWdlLWRpc2pvaW50IC8gTE9CTyBzcGxpdCAgICAgICAgICAoZGF0YXNldC5hc3NpZ25fc3BsaXRzKSAgIOKAlCBraWxscyA4NiUgbGVhawogICogc3ViLWNlbnRlciBBcmNGYWNlLCBLIHN1Yi1jZW50ZXJzICAgKG1vZGVsLlN1YkNlbnRlckFyY01hcmdpbinigJQgbm9pc3kgbGFiZWxzCiAgKiBjbGFzcy1iYWxhbmNlZCBvciBjb25mdXNpb24gYmF0Y2hlcyAoZGF0YXNldCBzYW1wbGVycykgICAgICAgIOKAlCB0YWlsICsgaGFyZC1uZWcKICAqIHRpZXItd2VpZ2h0ZWQgbG9zcyAoR09MRD5TSUxWRVI+RkQpICgtLXctc2lsdmVyLy0tdy1mZCkgICAgICAg4oCUIGRlLWNpcmN1bGFyCiAgKiBTQU0gZmxhdC1taW5pbWEgKyBvcHRpb25hbCBTV0EgICAgICAoc2FtLlNBTSAvIHN3YV91dGlscykgICAgIOKAlCByYWlzZXMgZXJyb3ItQVVDCiAgKiBsYWJlbCBzbW9vdGhpbmcgICAgICAgICAgICAgICAgICAgICAoLS1zbW9vdGgpICAgICAgICAgICAgICAgIOKAlCBjYWxpYnJhdGlvbgoKU2F2ZXMgdGhlIHJhdyB0cmFpbmluZyBjaGVja3BvaW50IChzdWItY2VudGVyIGhlYWQga2VwdCB3aG9sZSkuIFJ1bgpleHBvcnRfY2hlY2twb2ludC5weSBhZnRlcndhcmRzIHRvIGNvbGxhcHNlIEvihpIxIGludG8gYW4gaW5mZXIucHktY29tcGF0aWJsZSBiZXN0LnB0LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKZnJvbSB0b3JjaC51dGlscy5kYXRhIGltcG9ydCBEYXRhTG9hZGVyLCBXZWlnaHRlZFJhbmRvbVNhbXBsZXIKCmZyb20gZGF0YXNldCBpbXBvcnQgKE5vbUNyb3BEYXRhc2V0LCBhc3NpZ25fc3BsaXRzLCBjbGFzc19iYWxhbmNlZF93ZWlnaHRzLAogICAgICAgICAgICAgICAgICAgICBDb25mdXNpb25CYXRjaFNhbXBsZXIpCmZyb20gbW9kZWwgaW1wb3J0IE5vbUVtYmVkZGVyLCBTdWJDZW50ZXJBcmNNYXJnaW4KZnJvbSBzYW0gaW1wb3J0IFNBTQoKSEVSRSA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQKCgpkZWYgX2RldmljZSgpOgogICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICByZXR1cm4gdG9yY2guZGV2aWNlKCJjdWRhIikKICAgIGlmIHRvcmNoLmJhY2tlbmRzLm1wcy5pc19hdmFpbGFibGUoKToKICAgICAgICByZXR1cm4gdG9yY2guZGV2aWNlKCJtcHMiKQogICAgcmV0dXJuIHRvcmNoLmRldmljZSgiY3B1IikKCgpkZWYgbG9hZF9tYW5pZmVzdChkYXRhX2RpcjogUGF0aCk6CiAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcihvcGVuKGRhdGFfZGlyIC8gIm1hbmlmZXN0LmNzdiIsIGVuY29kaW5nPSJ1dGYtOCIpKSkKICAgIGxhYmVscyA9IHNvcnRlZCh7clsibGFiZWwiXSBmb3IgciBpbiByb3dzfSkKICAgIGxhYjJpZHggPSB7YzogaSBmb3IgaSwgYyBpbiBlbnVtZXJhdGUobGFiZWxzKX0KICAgIHNpbSA9IHt9CiAgICBzcCA9IGRhdGFfZGlyIC8gInNpbWlsYXJfbWFwLmpzb24iCiAgICBpZiBzcC5leGlzdHMoKToKICAgICAgICByYXcgPSBqc29uLmxvYWQob3BlbihzcCwgZW5jb2Rpbmc9InV0Zi04IikpCiAgICAgICAgc2ltID0ge2xhYjJpZHhba106IFtsYWIyaWR4W3ZdIGZvciB2IGluIHZzIGlmIHYgaW4gbGFiMmlkeF0KICAgICAgICAgICAgICAgZm9yIGssIHZzIGluIHJhdy5pdGVtcygpIGlmIGsgaW4gbGFiMmlkeH0KICAgIHJldHVybiByb3dzLCBsYWJlbHMsIGxhYjJpZHgsIHNpbQoKCmRlZiBidWlsZF9sb2FkZXJzKGFyZ3MsIGRhdGFfZGlyOiBQYXRoKToKICAgIHJvd3MsIGxhYmVscywgbGFiMmlkeCwgc2ltID0gbG9hZF9tYW5pZmVzdChkYXRhX2RpcikKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKICAgIGRmID0gcGQuRGF0YUZyYW1lKHJvd3MpCiAgICBkZlsic3BsaXQiXSA9IGFzc2lnbl9zcGxpdHMoZGYsIG1vZGU9YXJncy5zcGxpdCwgaG9sZG91dF9ib29rPWFyZ3MuaG9sZG91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWxfZnJhYz1hcmdzLnZhbF9mcmFjLCB0ZXN0X2ZyYWM9YXJncy50ZXN0X2ZyYWMsIHNlZWQ9YXJncy5zZWVkKQogICAgdGllcl93ID0geyJHT0xEIjogMS4wLCAiU0lMVkVSIjogYXJncy53X3NpbHZlciwgIkZEIjogYXJncy53X2ZkfQoKICAgIGRlZiBzdWJzZXQoc3BsaXQsIHNvdXJjZXMpOgogICAgICAgIGQgPSBkZlsoZGZbInNwbGl0Il0gPT0gc3BsaXQpICYgKGRmWyJzb3VyY2UiXS5pc2luKHNvdXJjZXMpKV0KICAgICAgICBwYXRocyA9IFtzdHIoZGF0YV9kaXIgLyBwKSBmb3IgcCBpbiBkWyJwYXRoIl1dCiAgICAgICAgeSA9IFtsYWIyaWR4W2NdIGZvciBjIGluIGRbImxhYmVsIl1dCiAgICAgICAgdyA9IFt0aWVyX3cuZ2V0KHQsIDEuMCkgZm9yIHQgaW4gZFsidGllciJdXQogICAgICAgIHJldHVybiBwYXRocywgeSwgdwoKICAgIHRyX3AsIHRyX3ksIHRyX3cgPSBzdWJzZXQoInRyYWluIiwgeyJjcm9wIiwgImZkIn0pCiAgICB2YV9wLCB2YV95LCB2YV93ID0gc3Vic2V0KCJ2YWwiLCB7ImNyb3AifSkKICAgIHRyID0gTm9tQ3JvcERhdGFzZXQodHJfcCwgdHJfeSwgaW1nPWFyZ3MuaW1nLCB0cmFpbj1UcnVlLCB3ZWlnaHRzPXRyX3cpCiAgICB2YSA9IE5vbUNyb3BEYXRhc2V0KHZhX3AsIHZhX3ksIGltZz1hcmdzLmltZywgdHJhaW49RmFsc2UsIHdlaWdodHM9dmFfdykKCiAgICBpZiBhcmdzLnNhbXBsZXIgPT0gImNvbmZ1c2lvbiIgYW5kIHNpbToKICAgICAgICBicyA9IENvbmZ1c2lvbkJhdGNoU2FtcGxlcih0cl95LCBzaW0sIGJhdGNoX3NpemU9YXJncy5iYXRjaCwgcGVyX2NsYXNzPWFyZ3MucGVyX2NsYXNzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlZWQ9YXJncy5zZWVkLCBsZW5ndGg9bWF4KDEsIGxlbih0cl95KSAvLyBhcmdzLmJhdGNoKSkKICAgICAgICB0bCA9IERhdGFMb2FkZXIodHIsIGJhdGNoX3NhbXBsZXI9YnMsIG51bV93b3JrZXJzPWFyZ3Mud29ya2VycywgcGluX21lbW9yeT1UcnVlKQogICAgZWxzZToKICAgICAgICBzYW1wID0gV2VpZ2h0ZWRSYW5kb21TYW1wbGVyKGNsYXNzX2JhbGFuY2VkX3dlaWdodHModHJfeSwgbGVuKGxhYmVscykpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbnVtX3NhbXBsZXM9bGVuKHRyX3kpLCByZXBsYWNlbWVudD1UcnVlKQogICAgICAgIHRsID0gRGF0YUxvYWRlcih0ciwgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoLCBzYW1wbGVyPXNhbXAsCiAgICAgICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPWFyZ3Mud29ya2VycywgcGluX21lbW9yeT1UcnVlLCBkcm9wX2xhc3Q9VHJ1ZSkKICAgIHZsID0gRGF0YUxvYWRlcih2YSwgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoLCBzaHVmZmxlPUZhbHNlLAogICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPWFyZ3Mud29ya2VycywgcGluX21lbW9yeT1UcnVlKQogICAgcHJpbnQoZiJbZGF0YV0gY2xhc3Nlcz17bGVuKGxhYmVscyl9IHRyYWluPXtsZW4odHJfeSl9IHZhbD17bGVuKHZhX3kpfSAiCiAgICAgICAgICBmInNhbXBsZXI9e2FyZ3Muc2FtcGxlcn0gc2ltLWdyb3Vwcz17bGVuKHNpbSl9IikKICAgIHJldHVybiB0bCwgdmwsIGxhYmVscywgbGFiMmlkeAoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGV2YWx1YXRlKGVtYiwgaGVhZCwgdmwsIGRldmljZSk6CiAgICBlbWIuZXZhbCgpOyBoZWFkLmV2YWwoKQogICAgY29ycmVjdCA9IHRvdGFsID0gMAogICAgZm9yIHgsIHksIF8gaW4gdmw6CiAgICAgICAgeCwgeSA9IHgudG8oZGV2aWNlKSwgeS50byhkZXZpY2UpCiAgICAgICAgbG9naXRzID0gaGVhZChlbWIoeCkpICAgICAgICAgICAgICAgICAgICAgICAjIG5vIG1hcmdpbiAtPiBwbGFpbiBzdWItY2VudGVyIGNvc2luZSpzCiAgICAgICAgY29ycmVjdCArPSAobG9naXRzLmFyZ21heCgxKSA9PSB5KS5zdW0oKS5pdGVtKCkKICAgICAgICB0b3RhbCArPSB5Lm51bWVsKCkKICAgIHJldHVybiBjb3JyZWN0IC8gbWF4KDEsIHRvdGFsKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kYXRhIiwgZGVmYXVsdD1zdHIoSEVSRSAvICJkYXRhIikpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD1zdHIoSEVSRSAvICJjaGVja3BvaW50cyIpKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWFyY2giLCBkZWZhdWx0PSJyZXNuZXQxOCIsIGNob2ljZXM9WyJyZXNuZXQxOCIsICJyZXNuZXQzNCIsICJyZXNuZXQ1MCJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWVtYmVkLWRpbSIsIHR5cGU9aW50LCBkZWZhdWx0PTI1NikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1pbWciLCB0eXBlPWludCwgZGVmYXVsdD0xMjgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tayIsIHR5cGU9aW50LCBkZWZhdWx0PTMsIGhlbHA9IkFyY0ZhY2Ugc3ViLWNlbnRlcnMgcGVyIGNsYXNzIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zMC4wKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW0iLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMzApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MzApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2giLCB0eXBlPWludCwgZGVmYXVsdD0xMjgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbHIiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTNlLTQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td2QiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTVlLTQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc21vb3RoIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjEsIGhlbHA9ImxhYmVsIHNtb290aGluZyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2FtIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwgaGVscD0iU0FNIGZsYXQtbWluaW1hIChyZWNvbW1lbmRlZCkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJobyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wNSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zd2EiLCBhY3Rpb249InN0b3JlX3RydWUiLCBoZWxwPSJTV0Egb3ZlciB0aGUgbGFzdCBzd2EtZXBvY2hzIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zd2EtZXBvY2hzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zYW1wbGVyIiwgZGVmYXVsdD0iYmFsYW5jZWQiLCBjaG9pY2VzPVsiYmFsYW5jZWQiLCAiY29uZnVzaW9uIl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcGVyLWNsYXNzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGRlZmF1bHQ9InBhZ2VfZGlzam9pbnQiLCBjaG9pY2VzPVsicGFnZV9kaXNqb2ludCIsICJsb2JvIl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taG9sZG91dCIsIGRlZmF1bHQ9IiIsIGhlbHA9ImJvb2sgY29kZSBoZWxkIG91dCBmb3IgLS1zcGxpdCBsb2JvIChlLmcuIHN0dDQpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS12YWwtZnJhYyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4xKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRlc3QtZnJhYyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4xKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXctc2lsdmVyIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdy1mZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC40KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNlZWQiLCB0eXBlPWludCwgZGVmYXVsdD00MikKICAgICMgLS0tLSByZXN1bWUgLyBIdWdnaW5nIEZhY2Ugc3luYyAoc3Vydml2ZSBLYWdnbGUgc2Vzc2lvbiByZXNldHMpIC0tLS0KICAgIGFwLmFkZF9hcmd1bWVudCgiLS1oZi1yZXBvIiwgZGVmYXVsdD0iIiwgaGVscD0iSEYgbW9kZWwgcmVwbywgZS5nLiBtZG50NTcxL25vbS1lbWJlZC1hcmNmYWNlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1oZi10b2tlbiIsIGRlZmF1bHQ9IiIsIGhlbHA9IkhGIHRva2VuIChlbHNlIGVudiBIRl9UT0tFTiAvIEthZ2dsZSBTZWNyZXQpIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yZXN1bWUiLCBhY3Rpb249InN0b3JlX3RydWUiLCBkZWZhdWx0PVRydWUsCiAgICAgICAgICAgICAgICAgICAgaGVscD0icmVzdW1lIGZyb20gbG9jYWwvSEYgbGFzdC5wdCBpZiBwcmVzZW50IChkZWZhdWx0IE9OKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbm8tcmVzdW1lIiwgZGVzdD0icmVzdW1lIiwgYWN0aW9uPSJzdG9yZV9mYWxzZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHVzaC1ldmVyeSIsIHR5cGU9aW50LCBkZWZhdWx0PTEsIGhlbHA9InB1c2ggbGFzdC5wdCB0byBIRiBldmVyeSBOIGVwb2NocyIpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgdG9yY2gubWFudWFsX3NlZWQoYXJncy5zZWVkKTsgbnAucmFuZG9tLnNlZWQoYXJncy5zZWVkKQogICAgZGV2aWNlID0gX2RldmljZSgpCiAgICBQYXRoKGFyZ3Mub3V0KS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0bCwgdmwsIGxhYmVscywgbGFiMmlkeCA9IGJ1aWxkX2xvYWRlcnMoYXJncywgUGF0aChhcmdzLmRhdGEpKQogICAgbl9jbHMgPSBsZW4obGFiZWxzKQoKICAgIGVtYiA9IE5vbUVtYmVkZGVyKGFyZ3MuZW1iZWRfZGltLCBwcmV0cmFpbmVkPVRydWUsIGFyY2g9YXJncy5hcmNoKS50byhkZXZpY2UpCiAgICBoZWFkID0gU3ViQ2VudGVyQXJjTWFyZ2luKGFyZ3MuZW1iZWRfZGltLCBuX2Nscywgaz1hcmdzLmssIHM9YXJncy5zLCBtPWFyZ3MubSkudG8oZGV2aWNlKQogICAgcGFyYW1zID0gbGlzdChlbWIucGFyYW1ldGVycygpKSArIGxpc3QoaGVhZC5wYXJhbWV0ZXJzKCkpCgogICAgaWYgYXJncy5zYW06CiAgICAgICAgb3B0ID0gU0FNKHBhcmFtcywgdG9yY2gub3B0aW0uQWRhbVcsIHJobz1hcmdzLnJobywgbHI9YXJncy5sciwgd2VpZ2h0X2RlY2F5PWFyZ3Mud2QpCiAgICAgICAgc2NoZWQgPSB0b3JjaC5vcHRpbS5scl9zY2hlZHVsZXIuQ29zaW5lQW5uZWFsaW5nTFIob3B0LmJhc2Vfb3B0aW1pemVyLCBUX21heD1hcmdzLmVwb2NocykKICAgIGVsc2U6CiAgICAgICAgb3B0ID0gdG9yY2gub3B0aW0uQWRhbVcocGFyYW1zLCBscj1hcmdzLmxyLCB3ZWlnaHRfZGVjYXk9YXJncy53ZCkKICAgICAgICBzY2hlZCA9IHRvcmNoLm9wdGltLmxyX3NjaGVkdWxlci5Db3NpbmVBbm5lYWxpbmdMUihvcHQsIFRfbWF4PWFyZ3MuZXBvY2hzKQoKICAgIHN3YV9lbWIgPSB0b3JjaC5vcHRpbS5zd2FfdXRpbHMuQXZlcmFnZWRNb2RlbChlbWIpIGlmIGFyZ3Muc3dhIGVsc2UgTm9uZQoKICAgIGRlZiB3bG9zcyh4LCB5LCB3KToKICAgICAgICBsb2dpdHMgPSBoZWFkKGVtYih4KSwgeSkgICAgICAgICAgICAgICAgICAgICMgbWFyZ2luIG9uIHRhcmdldAogICAgICAgIHBlciA9IEYuY3Jvc3NfZW50cm9weShsb2dpdHMsIHksIGxhYmVsX3Ntb290aGluZz1hcmdzLnNtb290aCwgcmVkdWN0aW9uPSJub25lIikKICAgICAgICByZXR1cm4gKHBlciAqIHcpLm1lYW4oKQoKICAgICMgLS0tLSByZXN1bWUgKyBIdWdnaW5nIEZhY2Ugc3luYyAoc3Vydml2ZSBLYWdnbGUgcmVzZXRzKSAtLS0tLS0tLS0tLS0tLS0KICAgIGZyb20gaHViIGltcG9ydCByZXNvbHZlX3Rva2VuLCBlbnN1cmVfcmVwbywgcHVzaCBhcyBoZl9wdXNoLCBwdWxsIGFzIGhmX3B1bGwKICAgIHRva2VuID0gcmVzb2x2ZV90b2tlbihhcmdzLmhmX3Rva2VuKQogICAgaWYgYXJncy5oZl9yZXBvIGFuZCB0b2tlbjoKICAgICAgICBlbnN1cmVfcmVwbyhhcmdzLmhmX3JlcG8sIHRva2VuKQogICAgZWxpZiBhcmdzLmhmX3JlcG86CiAgICAgICAgcHJpbnQoIltoZl0gLS1oZi1yZXBvIHNldCBidXQgbm8gdG9rZW4gKEhGX1RPS0VOIC8gS2FnZ2xlIFNlY3JldCkgLT4gbG9jYWwtb25seSIpCiAgICBsYXN0X3BhdGggPSBQYXRoKGFyZ3Mub3V0KSAvICJsYXN0LnB0IgogICAgYmFzZV9vcHQgPSAobGFtYmRhOiBvcHQuYmFzZV9vcHRpbWl6ZXIpIGlmIGFyZ3Muc2FtIGVsc2UgKGxhbWJkYTogb3B0KQoKICAgIGRlZiBzYXZlX2Z1bGwoZXAsIGJlc3RfKToKICAgICAgICB0b3JjaC5zYXZlKHsiYmFja2JvbmUiOiBlbWIuc3RhdGVfZGljdCgpLCAiaGVhZCI6IGhlYWQuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICJvcHQiOiBiYXNlX29wdCgpLnN0YXRlX2RpY3QoKSwgInNjaGVkIjogc2NoZWQuc3RhdGVfZGljdCgpLAogICAgICAgICAgICAgICAgICAgICJzd2EiOiBzd2FfZW1iLnN0YXRlX2RpY3QoKSBpZiBzd2FfZW1iIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgICAgICAiZXBvY2giOiBlcCwgImJlc3QiOiBiZXN0XywgImNsYXNzZXMiOiBsYWIyaWR4LCAiayI6IGFyZ3MuaywKICAgICAgICAgICAgICAgICAgICAiYXJjaCI6IGFyZ3MuYXJjaCwgImVtYmVkX2RpbSI6IGFyZ3MuZW1iZWRfZGltLCAiaW1nIjogYXJncy5pbWd9LAogICAgICAgICAgICAgICAgICAgbGFzdF9wYXRoKQoKICAgIHN0YXJ0X2Vwb2NoLCBiZXN0ID0gMSwgLTEuMAogICAgaWYgYXJncy5yZXN1bWU6CiAgICAgICAgcnAgPSBzdHIobGFzdF9wYXRoKSBpZiBsYXN0X3BhdGguZXhpc3RzKCkgZWxzZSAoCiAgICAgICAgICAgIGhmX3B1bGwoYXJncy5oZl9yZXBvLCAibGFzdC5wdCIsIHRva2VuLCBhcmdzLm91dCkgaWYgYXJncy5oZl9yZXBvIGVsc2UgTm9uZSkKICAgICAgICBpZiBycCBhbmQgUGF0aChycCkuZXhpc3RzKCk6CiAgICAgICAgICAgIGNrID0gdG9yY2gubG9hZChycCwgbWFwX2xvY2F0aW9uPWRldmljZSkKICAgICAgICAgICAgaWYgbGVuKGNrLmdldCgiY2xhc3NlcyIsIHt9KSkgPT0gbl9jbHM6CiAgICAgICAgICAgICAgICBlbWIubG9hZF9zdGF0ZV9kaWN0KGNrWyJiYWNrYm9uZSJdKTsgaGVhZC5sb2FkX3N0YXRlX2RpY3QoY2tbImhlYWQiXSkKICAgICAgICAgICAgICAgIGJhc2Vfb3B0KCkubG9hZF9zdGF0ZV9kaWN0KGNrWyJvcHQiXSk7IHNjaGVkLmxvYWRfc3RhdGVfZGljdChja1sic2NoZWQiXSkKICAgICAgICAgICAgICAgIGlmIHN3YV9lbWIgYW5kIGNrLmdldCgic3dhIik6CiAgICAgICAgICAgICAgICAgICAgc3dhX2VtYi5sb2FkX3N0YXRlX2RpY3QoY2tbInN3YSJdKQogICAgICAgICAgICAgICAgc3RhcnRfZXBvY2gsIGJlc3QgPSBpbnQoY2tbImVwb2NoIl0pICsgMSwgZmxvYXQoY2suZ2V0KCJiZXN0IiwgLTEuMCkpCiAgICAgICAgICAgICAgICBwcmludChmIltyZXN1bWVdIHtycH0gLT4gY29udGludWUgYXQgZXBvY2gge3N0YXJ0X2Vwb2NofSAoYmVzdCB7YmVzdDouNGZ9KSIpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBwcmludChmIltyZXN1bWVdIGNsYXNzIG1pc21hdGNoICh7bGVuKGNrLmdldCgnY2xhc3Nlcycse30pKX0gdnMge25fY2xzfSkgLT4gZnJlc2giKQogICAgaWYgc3RhcnRfZXBvY2ggPiBhcmdzLmVwb2NoczoKICAgICAgICBwcmludChmIltyZXN1bWVdIGFscmVhZHkgYXQge2FyZ3MuZXBvY2hzfSBlcG9jaHMg4oCUIG5vdGhpbmcgdG8gZG8uIik7IHJldHVybgoKICAgIGZvciBlcCBpbiByYW5nZShzdGFydF9lcG9jaCwgYXJncy5lcG9jaHMgKyAxKToKICAgICAgICBlbWIudHJhaW4oKTsgaGVhZC50cmFpbigpCiAgICAgICAgcnVuID0gMC4wOyBuYiA9IDAKICAgICAgICBmb3IgeCwgeSwgdyBpbiB0bDoKICAgICAgICAgICAgeCwgeSwgdyA9IHgudG8oZGV2aWNlKSwgeS50byhkZXZpY2UpLCB3LmZsb2F0KCkudG8oZGV2aWNlKQogICAgICAgICAgICBpZiBhcmdzLnNhbToKICAgICAgICAgICAgICAgIGxvc3MgPSB3bG9zcyh4LCB5LCB3KTsgbG9zcy5iYWNrd2FyZCgpOyBvcHQuZmlyc3Rfc3RlcCh6ZXJvX2dyYWQ9VHJ1ZSkKICAgICAgICAgICAgICAgIHdsb3NzKHgsIHksIHcpLmJhY2t3YXJkKCk7IG9wdC5zZWNvbmRfc3RlcCh6ZXJvX2dyYWQ9VHJ1ZSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIG9wdC56ZXJvX2dyYWQoKTsgbG9zcyA9IHdsb3NzKHgsIHksIHcpOyBsb3NzLmJhY2t3YXJkKCk7IG9wdC5zdGVwKCkKICAgICAgICAgICAgcnVuICs9IGZsb2F0KGxvc3MuZGV0YWNoKCkpOyBuYiArPSAxCiAgICAgICAgc2NoZWQuc3RlcCgpCiAgICAgICAgcHJpbnQoZiJbZXAge2VwOjAyZH1dIHRyYWluIGxvc3M9e3J1biAvIG1heCgxLCBuYik6LjRmfSIsIGVuZD0iICAiKQogICAgICAgIGlmIGFyZ3Muc3dhIGFuZCBlcCA+IGFyZ3MuZXBvY2hzIC0gYXJncy5zd2FfZXBvY2hzOgogICAgICAgICAgICBzd2FfZW1iLnVwZGF0ZV9wYXJhbWV0ZXJzKGVtYikKICAgICAgICBhY2MgPSBldmFsdWF0ZShlbWIsIGhlYWQsIHZsLCBkZXZpY2UpCiAgICAgICAgcHJpbnQoZiJbZXAge2VwOjAyZH0ve2FyZ3MuZXBvY2hzfV0gdmFsIGhlYWQtdG9wMT17YWNjOi40Zn0gbHI9e3NjaGVkLmdldF9sYXN0X2xyKClbMF06LjJlfSIpCiAgICAgICAgaWYgYWNjID49IGJlc3Q6CiAgICAgICAgICAgIGJlc3QgPSBhY2MKICAgICAgICAgICAgdG9yY2guc2F2ZSh7ImJhY2tib25lIjogZW1iLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgICAgICAgICAgICAgImhlYWRfVyI6IGhlYWQuVy5kZXRhY2goKS5jcHUoKSwgImsiOiBhcmdzLmssCiAgICAgICAgICAgICAgICAgICAgICAgICJjbGFzc2VzIjogbGFiMmlkeCwgImFyY2giOiBhcmdzLmFyY2gsCiAgICAgICAgICAgICAgICAgICAgICAgICJlbWJlZF9kaW0iOiBhcmdzLmVtYmVkX2RpbSwgImltZyI6IGFyZ3MuaW1nLAogICAgICAgICAgICAgICAgICAgICAgICAidmFsX2hlYWRfdG9wMSI6IGFjY30sCiAgICAgICAgICAgICAgICAgICAgICAgUGF0aChhcmdzLm91dCkgLyAidHJhaW5fYmVzdC5wdCIpCiAgICAgICAgICAgIHByaW50KGYiICAgICAgICAgIC0+IHNhdmVkIHRyYWluX2Jlc3QucHQgKHZhbCBoZWFkLXRvcDEge2FjYzouNGZ9KSIpCiAgICAgICAgIyBmdWxsLXN0YXRlIHJlc3VtZSBjaGVja3BvaW50IGV2ZXJ5IGVwb2NoICsgcHVzaCB0byBIRiAoS2FnZ2xlLXJlc2V0IHNhZmUpCiAgICAgICAgc2F2ZV9mdWxsKGVwLCBiZXN0KQogICAgICAgIGlmIGFyZ3MuaGZfcmVwbyBhbmQgdG9rZW4gYW5kIChlcCAlIGFyZ3MucHVzaF9ldmVyeSA9PSAwIG9yIGVwID09IGFyZ3MuZXBvY2hzKToKICAgICAgICAgICAgaGZfcHVzaChsYXN0X3BhdGgsIGFyZ3MuaGZfcmVwbywgImxhc3QucHQiLCB0b2tlbikKICAgICAgICAgICAgaGZfcHVzaChQYXRoKGFyZ3Mub3V0KSAvICJ0cmFpbl9iZXN0LnB0IiwgYXJncy5oZl9yZXBvLCAidHJhaW5fYmVzdC5wdCIsIHRva2VuKQoKICAgIGlmIGFyZ3Muc3dhOgogICAgICAgIHRvcmNoLm9wdGltLnN3YV91dGlscy51cGRhdGVfYm4oCiAgICAgICAgICAgICgoeC50byhkZXZpY2UpLCkgZm9yIHgsIF8sIF8gaW4gdGwpLCBzd2FfZW1iLCBkZXZpY2U9ZGV2aWNlKQogICAgICAgIGFjYyA9IGV2YWx1YXRlKHN3YV9lbWIubW9kdWxlLCBoZWFkLCB2bCwgZGV2aWNlKQogICAgICAgIHByaW50KGYiW3N3YV0gdmFsIGhlYWQtdG9wMT17YWNjOi40Zn0iKQogICAgICAgIGlmIGFjYyA+PSBiZXN0OgogICAgICAgICAgICB0b3JjaC5zYXZlKHsiYmFja2JvbmUiOiBzd2FfZW1iLm1vZHVsZS5zdGF0ZV9kaWN0KCksCiAgICAgICAgICAgICAgICAgICAgICAgICJoZWFkX1ciOiBoZWFkLlcuZGV0YWNoKCkuY3B1KCksICJrIjogYXJncy5rLAogICAgICAgICAgICAgICAgICAgICAgICAiY2xhc3NlcyI6IGxhYjJpZHgsICJhcmNoIjogYXJncy5hcmNoLAogICAgICAgICAgICAgICAgICAgICAgICAiZW1iZWRfZGltIjogYXJncy5lbWJlZF9kaW0sICJpbWciOiBhcmdzLmltZywKICAgICAgICAgICAgICAgICAgICAgICAgInZhbF9oZWFkX3RvcDEiOiBhY2N9LCBQYXRoKGFyZ3Mub3V0KSAvICJ0cmFpbl9iZXN0LnB0IikKICAgICAgICAgICAgcHJpbnQoZiJbc3dhXSAtPiBzYXZlZCB0cmFpbl9iZXN0LnB0ICh2YWwgaGVhZC10b3AxIHthY2M6LjRmfSkiKQogICAgcHJpbnQoZiJbZG9uZV0gYmVzdCB2YWwgaGVhZC10b3AxPXtiZXN0Oi40Zn0iKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK",
    "export_checkpoint.py": "IiIiQ29sbGFwc2UgdGhlIHN1Yi1jZW50ZXIgaGVhZCBL4oaSMSBhbmQgd3JpdGUgYW4gaW5mZXIucHktY29tcGF0aWJsZSBiZXN0LnB0LgoKaW5mZXIuTm9tRW5jb2RlciBleHBlY3RzIGNrID0ge2JhY2tib25lLCBoZWFkOntXOihuX2NsYXNzZXMsIGVtYmVkKX0sIGNsYXNzZXMsCmFyY2gsIGVtYmVkX2RpbSwgaW1nfSBhbmQgbXVsdGlwbGllcyBoZWFkWyJXIl0gQCBlIGZvciB0aGUgaGVhZC1sb2dpdCBnYXRlIChQMCkuClN1Yi1jZW50ZXIgdHJhaW5pbmcgcHJvZHVjZWQgVyBvZiBzaGFwZSAobl9jbGFzc2VzKkssIGVtYmVkKS4gRm9yIGVhY2ggY2xhc3Mgd2UKa2VlcCB0aGUgRE9NSU5BTlQgc3ViLWNlbnRlciDigJQgdGhlIG9uZSBpdHMgcmVhbCBHT0xEIGNyb3BzIGFsaWduIHRvIG1vc3Qg4oCUIHNvIHRoZQpleHBvcnRlZCBwZXItY2xhc3MgdmVjdG9yIGlzIHRoZSBDTEVBTiBjZW50cm9pZCwgd2l0aCBub2lzeS9taXMtY3V0IHZhcmlhbnRzIGxlZnQKYmVoaW5kIG9uIHRoZSBkaXNjYXJkZWQgc3ViLWNlbnRlcnMgKHRoYXQgaXMgdGhlIHdob2xlIHBvaW50IG9mIHN1Yi1jZW50ZXJzKS4KCiAgICBweXRob24gQXJjRmFjZS9leHBvcnRfY2hlY2twb2ludC5weSAgICAgICAgICAgICMgZG9taW5hbnQgKGVtYmVkcyBHT0xEIGNyb3BzKQogICAgcHl0aG9uIEFyY0ZhY2UvZXhwb3J0X2NoZWNrcG9pbnQucHkgLS1jb2xsYXBzZSBtZWFuICAgIyBmYXN0LCBubyBlbWJlZGRpbmcKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgY3N2CmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCgpIRVJFID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudAppbXBvcnQgc3lzCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoSEVSRSkpCmZyb20gbW9kZWwgaW1wb3J0IE5vbUVtYmVkZGVyICAgICAgICAgICAgICAgICAgICAgICAjIG5vcWE6IEU0MDIKZnJvbSBkYXRhc2V0IGltcG9ydCBOb21Dcm9wRGF0YXNldCAgICAgICAgICAgICAgICAgICMgbm9xYTogRTQwMgoKCmRlZiBfZGV2aWNlKCk6CiAgICByZXR1cm4gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpCiAgICAgICAgICAgICAgICAgICAgICAgIGVsc2UgKCJtcHMiIGlmIHRvcmNoLmJhY2tlbmRzLm1wcy5pc19hdmFpbGFibGUoKSBlbHNlICJjcHUiKSkKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBkb21pbmFudF9zdWJjZW50ZXJzKGNrLCBkYXRhX2RpciwgZGV2aWNlLCBjYXA9NDApOgogICAgIiIiUGVyIGNsYXNzLCBwaWNrIGFyZ21heF9rIG1lYW4gY29zaW5lKEdPTEQgY3JvcHNfYywgc3ViY2VudGVyX3tjLGt9KS4iIiIKICAgIGxhYjJpZHggPSBja1siY2xhc3NlcyJdOyBuX2NscyA9IGxlbihsYWIyaWR4KTsgayA9IGNrWyJrIl0KICAgIGVtYiA9IE5vbUVtYmVkZGVyKGNrWyJlbWJlZF9kaW0iXSwgcHJldHJhaW5lZD1GYWxzZSwgYXJjaD1ja1siYXJjaCJdKS50byhkZXZpY2UpCiAgICBlbWIubG9hZF9zdGF0ZV9kaWN0KGNrWyJiYWNrYm9uZSJdKTsgZW1iLmV2YWwoKQogICAgVyA9IEYubm9ybWFsaXplKGNrWyJoZWFkX1ciXS50byhkZXZpY2UpLmZsb2F0KCksIGRpbT0xKS52aWV3KG5fY2xzLCBrLCAtMSkgICMgKEMsSyxFKQoKICAgIGJ5X2NscyA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3IgciBpbiBjc3YuRGljdFJlYWRlcihvcGVuKFBhdGgoZGF0YV9kaXIpIC8gIm1hbmlmZXN0LmNzdiIsIGVuY29kaW5nPSJ1dGYtOCIpKToKICAgICAgICBpZiByWyJzb3VyY2UiXSA9PSAiY3JvcCIgYW5kIHJbInRpZXIiXSA9PSAiR09MRCIgYW5kIHJbImxhYmVsIl0gaW4gbGFiMmlkeDoKICAgICAgICAgICAgYnlfY2xzW2xhYjJpZHhbclsibGFiZWwiXV1dLmFwcGVuZChzdHIoUGF0aChkYXRhX2RpcikgLyByWyJwYXRoIl0pKQoKICAgIGNob3NlbiA9IHRvcmNoLnplcm9zKG5fY2xzLCBkdHlwZT10b3JjaC5sb25nKQogICAgZm9yIGMgaW4gcmFuZ2Uobl9jbHMpOgogICAgICAgIHBhdGhzID0gYnlfY2xzLmdldChjLCBbXSlbOmNhcF0KICAgICAgICBpZiBub3QgcGF0aHM6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIGNyb3AtbGVzcyBjbGFzcyAtPiBzdWItY2VudGVyIDAKICAgICAgICAgICAgY29udGludWUKICAgICAgICBkcyA9IE5vbUNyb3BEYXRhc2V0KHBhdGhzLCBbY10gKiBsZW4ocGF0aHMpLCBpbWc9Y2tbImltZyJdLCB0cmFpbj1GYWxzZSkKICAgICAgICBFID0gdG9yY2guc3RhY2soW2RzW2ldWzBdIGZvciBpIGluIHJhbmdlKGxlbihkcykpXSkudG8oZGV2aWNlKQogICAgICAgIGUgPSBlbWIoRSkgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIChuLCBFKSBMMi1ub3JtZWQKICAgICAgICAjIG1lYW4gY29zaW5lIG9mIHRoaXMgY2xhc3MncyBjcm9wcyB0byBlYWNoIG9mIGl0cyBLIHN1Yi1jZW50ZXJzCiAgICAgICAgc2NvcmUgPSAoZSBAIFdbY10udCgpKS5tZWFuKDApICAgICAgICAgICAgICAgICMgKEssKQogICAgICAgIGNob3NlbltjXSA9IGludChzY29yZS5hcmdtYXgoKSkKICAgIFdjID0gdG9yY2guc3RhY2soW1dbYywgY2hvc2VuW2NdXSBmb3IgYyBpbiByYW5nZShuX2NscyldKSAgICMgKEMsIEUpCiAgICByZXR1cm4gV2MuY3B1KCksIGNob3NlbgoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ja3B0IiwgZGVmYXVsdD1zdHIoSEVSRSAvICJjaGVja3BvaW50cyIgLyAidHJhaW5fYmVzdC5wdCIpKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCBkZWZhdWx0PXN0cihIRVJFIC8gImRhdGEiKSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1vdXQiLCBkZWZhdWx0PXN0cihIRVJFIC8gImNoZWNrcG9pbnRzIiAvICJiZXN0LnB0IikpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY29sbGFwc2UiLCBkZWZhdWx0PSJkb21pbmFudCIsIGNob2ljZXM9WyJkb21pbmFudCIsICJtZWFuIl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taGYtcmVwbyIsIGRlZmF1bHQ9IiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taGYtdG9rZW4iLCBkZWZhdWx0PSIiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGNrID0gdG9yY2gubG9hZChhcmdzLmNrcHQsIG1hcF9sb2NhdGlvbj0iY3B1IikKICAgIG5fY2xzID0gbGVuKGNrWyJjbGFzc2VzIl0pOyBrID0gY2tbImsiXQogICAgaWYgYXJncy5jb2xsYXBzZSA9PSAibWVhbiIgb3IgayA9PSAxOgogICAgICAgIFcgPSBGLm5vcm1hbGl6ZShja1siaGVhZF9XIl0uZmxvYXQoKS52aWV3KG5fY2xzLCBrLCAtMSkubWVhbigxKSwgZGltPTEpCiAgICBlbHNlOgogICAgICAgIFcsIGNob3NlbiA9IGRvbWluYW50X3N1YmNlbnRlcnMoY2ssIGFyZ3MuZGF0YSwgX2RldmljZSgpKQogICAgICAgIHByaW50KGYiW2V4cG9ydF0gZG9taW5hbnQgc3ViLWNlbnRlciBwaWNrZWQgZm9yIHtuX2Nsc30gY2xhc3NlcyAiCiAgICAgICAgICAgICAgZiIobm9uLXplcm8gcGlja3M6IHtpbnQoKGNob3NlbiA+IDApLnN1bSgpKX0pIikKCiAgICBvdXQgPSB7ImJhY2tib25lIjogY2tbImJhY2tib25lIl0sICJoZWFkIjogeyJXIjogV30sICJjbGFzc2VzIjogY2tbImNsYXNzZXMiXSwKICAgICAgICAgICAiYXJjaCI6IGNrWyJhcmNoIl0sICJlbWJlZF9kaW0iOiBja1siZW1iZWRfZGltIl0sICJpbWciOiBja1siaW1nIl19CiAgICB0b3JjaC5zYXZlKG91dCwgYXJncy5vdXQpCiAgICBwcmludChmIltleHBvcnRdIHdyb3RlIHthcmdzLm91dH0gIGhlYWQuVz17dHVwbGUoVy5zaGFwZSl9ICAiCiAgICAgICAgICBmInZhbF9oZWFkX3RvcDE9e2NrLmdldCgndmFsX2hlYWRfdG9wMScpfSIpCiAgICBpZiBhcmdzLmhmX3JlcG86CiAgICAgICAgZnJvbSBodWIgaW1wb3J0IHJlc29sdmVfdG9rZW4sIGVuc3VyZV9yZXBvLCBwdXNoIGFzIGhmX3B1c2gKICAgICAgICB0b2sgPSByZXNvbHZlX3Rva2VuKGFyZ3MuaGZfdG9rZW4pCiAgICAgICAgaWYgdG9rOgogICAgICAgICAgICBlbnN1cmVfcmVwbyhhcmdzLmhmX3JlcG8sIHRvaykKICAgICAgICAgICAgaGZfcHVzaChhcmdzLm91dCwgYXJncy5oZl9yZXBvLCAiYmVzdC5wdCIsIHRvaykgICAjIGRlcGxveSBhcnRpZmFjdCBvbiBIRiB0b28KICAgIHByaW50KCJbZXhwb3J0XSBkcm9wLWluOiBjb3B5IHRvIHBpcGVsaW5lL2FsaWduX2VuZ2luZS9ub20tZW1iZWQvYmVzdC5wdCAiCiAgICAgICAgICAiKG9yIHBvaW50IE5vbUVuY29kZXIgY2twdCBhdCBpdCkuIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
    "evaluate.py": "IiIiSG9uZXN0IGV2YWx1YXRpb24gb2YgdGhlIGV4cG9ydGVkIGJlc3QucHQgb24gdGhlIFBBR0UtRElTSk9JTlQgdGVzdCBzcGxpdC4KClJlcG9ydHMgdGhlIG51bWJlcnMgdGhlIG9sZCBlbmNvZGVyIGZhaWxlZCBvbjoKICAqIHJldHJpZXZhbEAxIC8gQDUgdmlhIHRoZSBoZWFkICDigJQgcmFua2luZyBxdWFsaXR5ICh3YXMgNzguMSUgb2ZmbGluZSkuCiAgKiBwcm94eSBlcnJvci1nYXRlIEFVQyDigJQgY2FuIE1MUyAvIEVuZXJneSB0ZWxsIHdoZW4gdGhlIGhlYWQgaXMgV1JPTkc/CiAgICBTcGxpdHMgdGVzdCBwcmVkaWN0aW9ucyBpbnRvIGhlYWQtY29ycmVjdCB2cyBoZWFkLXdyb25nIChhdXRvLWxhYmVsIGFzCiAgICByZWZlcmVuY2UpIGFuZCBtZWFzdXJlcyBBVUMgb2YgdGhlIG9wZW4tc2V0IHNjb3JlIGF0IHNlcGFyYXRpbmcgdGhlbS4gVGhpcyBpcwogICAgdGhlIFAxIG1ldHJpYzsgd2l0aCBIVU1BTiB2ZXJkaWN0cyAoR8SQMCkgcmVydW4gdGhlIHNhbWUgY29kZSB3aXRoIGEgdmVyZGljdAogICAgY29sdW1uIGZvciB0aGUgdHJ1ZSBlcnJvci1BVUMgKGF1dG8tbGFiZWwgcHJveHkgb3Zlci1zdGF0ZXMgaXQsIHNvIHRyZWF0IGFzCiAgICBhbiB1cHBlciBib3VuZCkuCiAgKiBNTFMgPSBNYXgtTG9naXQtU2NvcmUgKFZhemUgSUNMUicyMik7IEVuZXJneSA9IGxvZ3N1bWV4cCBvdmVyIGNsYXNzZXMKICAgIChMaXUgTmV1cklQUycyMCkg4oCUIHRoZSB0d28gY2FuZGlkYXRlIG9wZW4tc2V0IGdhdGVzLgpTYW1lIHNwbGl0IHNlZWQgYXMgdHJhaW4ucHksIHNvIHRlc3QgcGFnZXMgd2VyZSBuZXZlciB0cmFpbmVkIG9uLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubi5mdW5jdGlvbmFsIGFzIEYKCkhFUkUgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50CmltcG9ydCBzeXMKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihIRVJFKSkKZnJvbSBtb2RlbCBpbXBvcnQgTm9tRW1iZWRkZXIgICAgICAgICAgICAgICAgICAgICAgICMgbm9xYTogRTQwMgpmcm9tIGRhdGFzZXQgaW1wb3J0IE5vbUNyb3BEYXRhc2V0LCBhc3NpZ25fc3BsaXRzICAgIyBub3FhOiBFNDAyCgoKZGVmIF9hdWMoc2NvcmVzLCBpc19wb3MpOgogICAgIiIiQVVDIHRoYXQgYSBISUdIIHNjb3JlIG1hcmtzIGEgcG9zaXRpdmUgKGhlcmU6IGhlYWQtQ09SUkVDVCkuIE1hbm4tV2hpdG5leS4iIiIKICAgIHMgPSBucC5hc2FycmF5KHNjb3Jlcyk7IHkgPSBucC5hc2FycmF5KGlzX3BvcykuYXN0eXBlKGJvb2wpCiAgICBwLCBuID0gc1t5XSwgc1t+eV0KICAgIGlmIGxlbihwKSA9PSAwIG9yIGxlbihuKSA9PSAwOgogICAgICAgIHJldHVybiBmbG9hdCgibmFuIikKICAgIG9yZGVyID0gbnAuYXJnc29ydChzKTsgcmFua3MgPSBucC5lbXB0eV9saWtlKG9yZGVyLCBmbG9hdCk7IHJhbmtzW29yZGVyXSA9IG5wLmFyYW5nZSgxLCBsZW4ocykgKyAxKQogICAgcmV0dXJuIChyYW5rc1t5XS5zdW0oKSAtIGxlbihwKSAqIChsZW4ocCkgKyAxKSAvIDIpIC8gKGxlbihwKSAqIGxlbihuKSkKCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ja3B0IiwgZGVmYXVsdD1zdHIoSEVSRSAvICJjaGVja3BvaW50cyIgLyAiYmVzdC5wdCIpKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCBkZWZhdWx0PXN0cihIRVJFIC8gImRhdGEiKSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zcGxpdCIsIGRlZmF1bHQ9InBhZ2VfZGlzam9pbnQiLCBjaG9pY2VzPVsicGFnZV9kaXNqb2ludCIsICJsb2JvIl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0taG9sZG91dCIsIGRlZmF1bHQ9IiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdmFsLWZyYWMiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10ZXN0LWZyYWMiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuMSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYmF0Y2giLCB0eXBlPWludCwgZGVmYXVsdD0yNTYpCiAgICBhcmdzID0gYXAucGFyc2VfYXJncygpCgogICAgZGV2ID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpCiAgICAgICAgICAgICAgICAgICAgICAgZWxzZSAoIm1wcyIgaWYgdG9yY2guYmFja2VuZHMubXBzLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpKQogICAgY2sgPSB0b3JjaC5sb2FkKGFyZ3MuY2twdCwgbWFwX2xvY2F0aW9uPWRldikKICAgIGxhYjJpZHggPSBja1siY2xhc3NlcyJdOyBuX2NscyA9IGxlbihsYWIyaWR4KQogICAgZW1iID0gTm9tRW1iZWRkZXIoY2tbImVtYmVkX2RpbSJdLCBwcmV0cmFpbmVkPUZhbHNlLCBhcmNoPWNrWyJhcmNoIl0pLnRvKGRldikKICAgIGVtYi5sb2FkX3N0YXRlX2RpY3QoY2tbImJhY2tib25lIl0pOyBlbWIuZXZhbCgpCiAgICBXbiA9IEYubm9ybWFsaXplKGNrWyJoZWFkIl1bIlciXS50byhkZXYpLmZsb2F0KCksIGRpbT0xKSAgICAgIyAoQywgRSkKCiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCiAgICBkZiA9IHBkLkRhdGFGcmFtZShsaXN0KGNzdi5EaWN0UmVhZGVyKG9wZW4oUGF0aChhcmdzLmRhdGEpIC8gIm1hbmlmZXN0LmNzdiIsIGVuY29kaW5nPSJ1dGYtOCIpKSkpCiAgICBkZlsic3BsaXQiXSA9IGFzc2lnbl9zcGxpdHMoZGYsIG1vZGU9YXJncy5zcGxpdCwgaG9sZG91dF9ib29rPWFyZ3MuaG9sZG91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWxfZnJhYz1hcmdzLnZhbF9mcmFjLCB0ZXN0X2ZyYWM9YXJncy50ZXN0X2ZyYWMsIHNlZWQ9YXJncy5zZWVkKQogICAgdGUgPSBkZlsoZGZbInNwbGl0Il0gPT0gInRlc3QiKSAmIChkZlsic291cmNlIl0gPT0gImNyb3AiKSAmIChkZlsibGFiZWwiXS5pc2luKGxhYjJpZHgpKV0KICAgIHBhdGhzID0gW3N0cihQYXRoKGFyZ3MuZGF0YSkgLyBwKSBmb3IgcCBpbiB0ZVsicGF0aCJdXQogICAgeSA9IFtsYWIyaWR4W2NdIGZvciBjIGluIHRlWyJsYWJlbCJdXQogICAgaWYgbm90IHBhdGhzOgogICAgICAgIHByaW50KCJbZXZhbF0gdGVzdCBzcGxpdCBlbXB0eSAodHJ5IC0tc3BsaXQgbG9ibyAtLWhvbGRvdXQgc3R0NCkiKTsgcmV0dXJuCiAgICBkcyA9IE5vbUNyb3BEYXRhc2V0KHBhdGhzLCB5LCBpbWc9Y2tbImltZyJdLCB0cmFpbj1GYWxzZSkKICAgIGRsID0gdG9yY2gudXRpbHMuZGF0YS5EYXRhTG9hZGVyKGRzLCBiYXRjaF9zaXplPWFyZ3MuYmF0Y2gsIG51bV93b3JrZXJzPTQpCgogICAgdG9wMSA9IHRvcDUgPSB0b3QgPSAwCiAgICBtbHNfYWxsLCBlbmVyZ3lfYWxsLCBjb3JyZWN0X2FsbCA9IFtdLCBbXSwgW10KICAgIGZvciB4LCB5eSwgXyBpbiBkbDoKICAgICAgICB4LCB5eSA9IHgudG8oZGV2KSwgeXkudG8oZGV2KQogICAgICAgIGxvZ2l0cyA9IGVtYih4KSBAIFduLnQoKSAgICAgICAgICAgICAgICAgICAgIyAoQiwgQykgY29zaW5lIGxvZ2l0cyBpbiBbLTEsMV0KICAgICAgICB0MSA9IGxvZ2l0cy5hcmdtYXgoMSkKICAgICAgICB0b3AxICs9ICh0MSA9PSB5eSkuc3VtKCkuaXRlbSgpCiAgICAgICAgdG9wNSArPSAobG9naXRzLnRvcGsoNSwgMSkuaW5kaWNlcyA9PSB5eVs6LCBOb25lXSkuYW55KDEpLnN1bSgpLml0ZW0oKQogICAgICAgIHRvdCArPSB5eS5udW1lbCgpCiAgICAgICAgbWxzX2FsbCArPSBsb2dpdHMubWF4KDEpLnZhbHVlcy5jcHUoKS50b2xpc3QoKSAgICAgICAgICAgICAgICMgTWF4LUxvZ2l0LVNjb3JlCiAgICAgICAgZW5lcmd5X2FsbCArPSB0b3JjaC5sb2dzdW1leHAobG9naXRzICogMzAuMCwgMSkuY3B1KCkudG9saXN0KCkgICMgRW5lcmd5IChzPTMwKQogICAgICAgIGNvcnJlY3RfYWxsICs9ICh0MSA9PSB5eSkuY3B1KCkudG9saXN0KCkKCiAgICBhdWNfbWxzID0gX2F1YyhtbHNfYWxsLCBjb3JyZWN0X2FsbCkKICAgIGF1Y19lbiA9IF9hdWMoZW5lcmd5X2FsbCwgY29ycmVjdF9hbGwpCiAgICBwcmludChmIltldmFsXSBzcGxpdD17YXJncy5zcGxpdH17Jy8nICsgYXJncy5ob2xkb3V0IGlmIGFyZ3MuaG9sZG91dCBlbHNlICcnfSAgIgogICAgICAgICAgZiJ0ZXN0IGNyb3BzPXt0b3R9ICBjbGFzc2VzIHRvdWNoZWQ9e2xlbihzZXQoeSkpfS97bl9jbHN9IikKICAgIHByaW50KGYiW2V2YWxdIHJldHJpZXZhbEAxPXt0b3AxL3RvdDouNGZ9ICBANT17dG9wNS90b3Q6LjRmfSIpCiAgICBwcmludChmIltldmFsXSBwcm94eSBlcnJvci1nYXRlIEFVQyAoaGlnaCBzY29yZSA9PiBoZWFkIGNvcnJlY3QpOiAgIgogICAgICAgICAgZiJNTFM9e2F1Y19tbHM6LjNmfSAgRW5lcmd5PXthdWNfZW46LjNmfSIpCiAgICBwcmludCgiW2V2YWxdIE5PVEU6IHByb3h5IHVzZXMgYXV0by1sYWJlbHMgYXMgdHJ1dGggKHVwcGVyIGJvdW5kKS4gUmVydW4gd2l0aCAiCiAgICAgICAgICAiaHVtYW4gdmVyZGljdHMgKEfEkDApIGZvciB0aGUgcmVhbCBlcnJvci1BVUMuIikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==",
}
for _fn, _b in MODULES.items():
    with open(os.path.join(CODE_DIR, _fn), "wb") as _f:
        _f.write(base64.b64decode(_b))
print("Đã ghi", len(MODULES), "module ->", CODE_DIR)

### 3) Thiết lập — tìm DATA + token HF

In [ ]:
import sys, glob, subprocess
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "huggingface_hub"], check=False)
assert os.path.exists(os.path.join(CODE_DIR, "train.py")), "Chạy cell (2) 'Ghi mã nguồn' trước."
sys.path.insert(0, CODE_DIR)

def find(name, roots=("/kaggle/input", "/kaggle/working", ".")):
    for r in roots:
        hits = sorted(glob.glob(f"{r}/**/{name}", recursive=True))
        if hits:
            return hits[0]
    return None

data_path = find("manifest.csv")
assert data_path, "Không thấy manifest.csv — Add Data: gắn dataset ArcFace/data (đã prepare_data.py ở máy)."
DATA_DIR = os.path.dirname(data_path)
OUT = "/kaggle/working/checkpoints" if os.path.isdir("/kaggle/working") else "checkpoints"

tok = ""
try:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    tok = os.environ.get("HF_TOKEN", "")
os.environ["HF_TOKEN"] = tok or ""

import torch
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out :", OUT)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (bật GPU!)")
print("HF  : repo=", HF_REPO or "(none)", "| token=", "yes" if tok else "NO -> local-only, KHÔNG reset-safe")

### 4) Train — reset-safe (chạy lại cell này sau reset để tiếp tục)

In [ ]:
def run(args):
    print("\n$", " ".join(args), flush=True)
    subprocess.run([sys.executable] + args, cwd=CODE_DIR, check=True)

cmd = ["train.py", "--data", DATA_DIR, "--out", OUT,
       "--epochs", str(EPOCHS), "--batch", str(BATCH), "--k", str(K),
       "--sampler", SAMPLER, "--split", SPLIT, "--holdout", HOLDOUT]
if USE_SAM: cmd.append("--sam")
if USE_SWA: cmd.append("--swa")
if HF_REPO: cmd += ["--hf-repo", HF_REPO]
run(cmd)

### 5) Export → `best.pt` (drop-in cho infer.py) + đẩy HF

In [ ]:
ecmd = ["export_checkpoint.py", "--data", DATA_DIR,
        "--ckpt", f"{OUT}/train_best.pt", "--out", f"{OUT}/best.pt"]
if HF_REPO: ecmd += ["--hf-repo", HF_REPO]
run(ecmd)

### 6) Đánh giá (page-disjoint test): retrieval@1/@5 + proxy error-AUC

In [ ]:
run(["evaluate.py", "--data", DATA_DIR, "--ckpt", f"{OUT}/best.pt",
     "--split", SPLIT, "--holdout", HOLDOUT])

### Xong
- `best.pt` ở `/kaggle/working/checkpoints/` **và** trên HF `HF_REPO`.
- Tải về repo: `huggingface-cli download <HF_REPO> best.pt --local-dir nom-embed`
- **Reset session?** Run All lại — cell Train tự resume từ HF, chỉ chạy epoch còn thiếu.